# Step 3 — Raw data-quality audit of the anime and user tables

**Purpose.** Step 1 compared the raw data with the author's description. This stage looks inside the columns that the research questions will depend on, and fixes the `null` username problem found in step 1.

**How to run on Kaggle.** Add the *MyAnimeList Dataset* as input and run all cells. Parts 1–3 only read the two small tables and take seconds. Part 4 reads two columns of the 80M-row list table and takes a few minutes, and part 7B reads it once more (about 4 minutes); set `RUN_LISTS = False` in the setup cell to skip both.

**Reading the output.** Each part ends with an *Observations* cell that first states what the output shows and only then interprets it.

| Part | Question |
|---|---|
| 1 | Which real values does pandas silently turn into `NaN`? |
| 2 | Anime table: placeholders, impossible values, how reliable is `score`, what can a pre-airing model use? |
| 3 | User table: placeholders, impossible ages and dates, suspicious aggregates |
| 4 | Do the per-user counts in `UserList` agree with the rows in `UserAnimeList`? |
| 5 | The users with no row in the list table: do they have other values, which would mean the list table is missing them? |
| 6 | Clarity checks: numbers that the first run did not print, and hypotheses that were never tested |
| 7 | Checks that parts 1 to 6 did not make: the list table, numeric columns, usernames, duplicates and contradictions inside a row |

**Where this step sits.** Step 3 of the exploration order (see [docs/exploration_order.md](../docs/exploration_order.md)). This notebook only **finds and counts** problems. It does not clean anything and it does not compare columns statistically. Cleaning rules are decided in step 4, from the evidence found here.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
RUN_LISTS = True  # False skips part 4, which reads 80M rows

print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Number of CSV files:", len(paths))

Python: 3.13.18 | pandas: 2.3.3
Number of CSV files: 9


## 1. Real values that pandas turns into `NaN`

By default pandas reads strings such as `null`, `NA` or `None` as missing. In step 1 this made the username `null` disappear. Here every cell of the two small tables is read as raw text, and the cells that pandas would have turned into `NaN` are counted per column. Both tables are then loaded again with only the empty string treated as missing.

### 1a. First look at the values, before counting the hidden missing words (NEW, added after the first run)

**Why this comes first.** Below, we count the words that pandas hides (`null`, `N/A`, `None`, ...) with a list of words that we wrote ourselves. In part 2 we also count placeholders with a list we wrote (`Unknown`, `Not available`, `None`, `?`, `0000-00-00`). Both lists come from what we happened to see, so they can miss other words, and they only find a cell that **equals** one of the words. A cell such as `Fridays at Unknown` is not found. So before any counting, we let the data show us which placeholder-like values exist.

**What the cells do.** The first one loads the two small tables with the safe load. For every text column of the anime table and the user table it counts three things, and prints the values behind each count:
- **Exact candidates:** a cell that is only a placeholder-like word, a symbol or a zero (`?`, `-`, `0`, `none`, `null`, `n/a`, `unknown`, `tba`, `tbd`, `undefined`, `not available`, `0000-00-00`, spaces only), in any capital letters.
- **Partial candidates:** a longer text that contains `unknown`, `not available`, `n/a`, `tba` or `tbd` (for example `Fridays at Unknown`).
- **Very short texts:** two characters or fewer, which are often a typo or a symbol.

The second cell prints the five most frequent values of every text column, so that we can read them with our own eyes: a placeholder is usually one of the most frequent values of its column.

**Limits.** This still looks for patterns that we think of. A placeholder written in an unexpected way (`idk`, `nothing`, `Earth` in `location`) can only be found by reading the most frequent values. The list table (80 million rows) is not scanned here; its text columns are `my_tags` (free text) and the two dates, which part 3 of notebook 02 already inspected.

**Read every hit before calling it a placeholder.** The scan only finds candidates. For example, the username `null` is a real user, so it will appear as an exact candidate in `username` although it is not a placeholder.

**What we do with the result.** If the output shows words that are not in our lists, we add them to the lists below and in part 2 before counting.


In [2]:
# NEW: placeholder-like cells in the text columns of the anime and user tables
import re

na_kwargs = dict(keep_default_na=False, na_values=[""])  # only an empty cell is missing
anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)

CANDIDATE = r"^\s*(?:|\?+|-+|\.+|_+|0|none|null|nan|n/?a|unknown|unk|tba|tbd|undefined|not available|not specified|no data|0000-00-00)\s*$"
PARTIAL = r"unknown|not available|n/a|tba|tbd"

def scan_text(df, label):
    print("=====", label)
    rows, details = [], {}
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        s = df[col].dropna().astype(str)
        exact = s[s.str.match(CANDIDATE, flags=re.IGNORECASE)]
        partial = s[s.str.contains(PARTIAL, flags=re.IGNORECASE, regex=True)]
        partial = partial[~partial.index.isin(exact.index)]
        short = s[(s.str.len() <= 2) & ~s.index.isin(exact.index)]
        rows.append({"column": col, "non_missing": len(s), "exact": len(exact), "partial": len(partial), "short": len(short)})
        details[col] = (exact, partial, short)
    table = pd.DataFrame(rows).set_index("column")
    hits = table[(table[["exact", "partial", "short"]] > 0).any(axis=1)]
    print(hits.sort_values("exact", ascending=False).to_string())
    for col in hits.index:
        exact, partial, short = details[col]
        print(f"\n{col}")
        if len(exact): print("  exact  :", exact.value_counts().head(6).to_dict())
        if len(partial): print("  partial:", partial.value_counts().head(6).to_dict())
        if len(short): print("  short  :", short.value_counts().head(6).to_dict())

scan_text(anime, "anime table")
print()
scan_text(users, "user table")


===== anime table
                non_missing  exact  partial  short
column                                            
source                14478   4210        0      0
broadcast              4271   2241       78      0
rating                14478    544        0      0
duration              14478    405        0      0
aired_string          14478    223        0      0
type                  14478     30        0   4271
title_synonyms         8937      0       16     11
title_japanese        14443      0        1    165
title                 14478      0        2     12
title_english          5724      0        9      3
background             1057      0        2      0
related               14478      0        1   4515
producer               8288      0        9      6
studio                 8544      0        0      3
opening_theme         14478      0        4   9784
ending_theme          14478      0        5   8807

title
  partial: {'The Epic Of ZektBach': 1, 'GetBackers': 1}
 

In [3]:
# NEW: the five most frequent values of every text column (read them: placeholders are often among them)
def top_values(df, label, n=5):
    print("=====", label)
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        counts = df[col].value_counts().head(n)
        print(f"{col}: " + " | ".join(f"{str(v)[:30]!r} {c:,}" for v, c in counts.items()))

top_values(anime, "anime table")
print()
top_values(users, "user table")


===== anime table
title: 'Hinamatsuri' 2 | 'Mushishi' 1 | 'Macross Dynamite 7' 1 | 'Ashita no Joe (Movie)' 1 | 'Decorator' 1
title_english: 'Cyborg 009' 4 | 'Digimon: The Movie' 3 | 'Big Windup!' 3 | 'Fight! Kickers' 3 | 'The Heroic Legend of Arslan' 3
title_japanese: 'ゲゲゲの鬼太郎' 6 | 'うっかりペネロペ' 4 | '아기공룡 둘리' 4 | '宇宙の騎士テッカマンブレード' 4 | 'ぼのぼの' 4
title_synonyms: 'Minna no Uta' 189 | 'Irodorimidori' 23 | 'Sekai Meisaku Gekijou' 5 | 'Doutoku Meisaku Series' 5 | 'Backkom, Bernard' 4
image_url: 'https://myanimelist.cdn-dena.c' 1 | 'https://myanimelist.cdn-dena.c' 1 | 'https://myanimelist.cdn-dena.c' 1 | 'https://myanimelist.cdn-dena.c' 1 | 'https://myanimelist.cdn-dena.c' 1
type: 'TV' 4,271 | 'OVA' 3,612 | 'Movie' 2,624 | 'Special' 1,948 | 'ONA' 1,144
source: 'Unknown' 4,210 | 'Original' 3,368 | 'Manga' 3,120 | 'Visual novel' 888 | 'Game' 607
status: 'Finished Airing' 13,791 | 'Currently Airing' 351 | 'Not yet aired' 336
aired_string: 'Not available' 223 | '2005' 47 | '2004' 43 | 'Jul, 2018 to ?'

### Observations for the first look

**How to read it.** The first cell counts, in every text column, the cells that are only a placeholder-like word or symbol (*exact*), the longer texts that contain such a word (*partial*) and the texts of two characters or fewer (*short*). The second cell prints the five most frequent values of each column. Many hits are false alarms, so every line below says what we checked by eye.

**1. The exact placeholder words of the anime table: our list was complete.**

| Column | Placeholder | Cells |
|---|---|---|
| `source` | `Unknown` | 4,210 |
| `broadcast` | `Unknown` | 2,241 |
| `rating` | `None` | 544 |
| `duration` | `Unknown` | 405 |
| `aired_string` | `Not available` | 223 |
| `type` | `Unknown` | 30 |

These are the same six columns and the same counts as the list of part 2. The scan found **no other word** (no `-`, `?`, `N/A`, `0` or spaces) in any text column of the anime table, and the five most frequent values of each column show none either. So the `PLACEHOLDERS` list of part 2 is enough for the anime table.

**2. Partial placeholders: `broadcast` with a weekday and no time.** 78 cells look like `Sundays at Unknown` (15), `Tuesdays at Unknown` (13), `Fridays at Unknown` (13), `Mondays at Unknown` (11), `Saturdays at Unknown` (11), `Thursdays at Unknown` (10), and a few more. The weekday is known and the time is not. Another 65 cells say `Not scheduled once per week`, which is no fixed weekday. So the time of the broadcast is unknown for 12,526 anime (86.5%): 10,207 empty cells, 2,241 `Unknown` and 78 "at Unknown".

**The other partial hits are false alarms.** The pattern `tba` appears inside ordinary words (`GetBackers`, `SoftBank`, `Basketball`, `Football`), and `unknown` appears in real titles (`The Unknown Girlfriend`) and song titles (`Destination Unknown`). Two opening themes have an artist called `Unknown`, which is negligible.

**3. Empty lists and empty dictionaries are placeholders that no word scan finds.** They are real text, so `isna()` does not see them either.

| Column | Value | Anime | Share of 14,478 |
|---|---|---|---|
| `opening_theme` | `[]` | 9,784 | 67.6% |
| `ending_theme` | `[]` | 8,807 | 60.8% |
| `related` | `[]` | 4,515 | 31.2% |
| `aired` | `{'from': None, 'to': None}` | 1,691 | 11.7% |
| `opening_theme` | `['None']` | 7 | 0.05% |

- **The theme columns are empty for about two thirds of the anime**, so they are not usable as they are. At best a yes/no feature ("has an opening") remains.
- **`aired` has no date at all for 1,691 anime, but `aired_string` is `Not available` for only 223.** So `aired_string` must still hold something for about 1,468 of them. Its most frequent values are a bare year (`2005` for 47 anime, `2004` for 43, `2000` for 40) and a month (`Jul, 2018 to ?` for 41 anime). Our reading, **tested in cell 1b below**, is that `aired` becomes empty when the date is only partial (a year or a month). If this is right, a release-year analysis must read `aired_string` and not `aired`, because `aired` loses 11.7% of the anime.
- **Consequence for notebook 02.** There we counted 147 anime labelled `Not yet aired` with no start date, using `aired`. Some of them probably have a month-level date such as `Jul, 2018`, which lies after the crawl and fits "not yet aired". This is checked in cell 1b below (check 4).

**4. Start dates that may carry a default day (hypothesis).** In `aired`, three of the most frequent start dates are on the 11th of different months: `2018-05-11` (57 anime: 39 plus 18), `2018-07-11` (24) and `2018-10-11` (15). A real calendar has no reason to favor the 11th. It looks like an automatic day for dates that are only known by month. This is tested in cell 1b below (check 2).

**5. HTML codes left in titles.** Several `title_english` and `title_synonyms` texts contain `&#039;` instead of an apostrophe (`Kuroko&#039;s Basketball`, `Kuroko&#039;s Basketball Special`). We saw them only because these titles also matched the `tba` pattern by chance, so we did not know the number of affected cells. Cell 1b below counts them. Titles must be decoded (`html.unescape`) before any matching or word count.

**6. User table.**
- **`location`: 309 cells are only a placeholder** out of 156,799 filled cells (0.2%): `Unknown` 61, `-` 42, `...` 28, `?` 23, `???` 20, `unknown` 18, and others not shown. 78 more cells contain the word inside a longer text (`Parts Unknown` 8, `Unknown Location` 2, `unknown, Poland` 1). The 26 words of the hidden-NA check below (`N/A`, `n/a`, `NA`, `None`) are only a part of these 309, so that list of words is far too short for `location`. Part 2 does not cover the user table, so these need their own list.
- **2,155 locations have two characters:** `UK` 668, `US` 109, `CA` 82, `NY` 70, `NJ` 57, `PA` 51. They are real values (abbreviations), but `CA` can mean Canada or California, so the grouping step needs rules.
- **15 usernames look like a placeholder** (`Unknown`, `unk`, `----`, `________`, `null`, ...), but usernames are unique and each one is a real user. **No username is treated as missing.** The 194 partial hits are normal names (`UnknownTitanGuy`).
- **`birth_date`: January 1 looks like a default day.** The five most frequent dates are all January 1: `1992-01-01` 630 users, `1993-01-01` 625, `1991-01-01` 599, `1994-01-01` 587 and `1990-01-01` 574. If the 168,741 dates were spread over the year, each date would have about 18 users. So January 1 is about 30 times too frequent, which fits "only the year was given" (a hypothesis, tested in cell 1b below). Then the age is correct to the year only, a month-of-birth analysis is meaningless, and the birth year 1930 of part 6 is probably the same default.
- **`last_online`: a spike, not a placeholder.** The five most frequent values all lie on 2013-09-13 between 12:52 and 13:38, with 15 to 19 users in one minute. Something happened to many accounts at that moment. We do not know what.
- **`gender` has three clean labels** and no placeholder.

**What this changes.**
- The `PLACEHOLDERS` list of part 2 stays as it is for the anime table.
- Added to the problems of step 4: the empty lists and the `None` dictionaries, the weekday without a time, the placeholders in `location`, the January 1 birth dates and the HTML codes (rows 17 to 22 of the summary below).
- **The four checks** (HTML codes, day of the month in `aired`, share of January 1, `aired` against `aired_string`) were added after this run. Their results are in cell 1b below.


### 1b. Four checks on the guesses of part 1a (NEW, added after the second run)

**Why.** The first look (1a) produced four guesses that we wrote down as hypotheses. Each one can be turned into a number with the two small tables that are already loaded, so we test them here before we rely on them.

| # | Guess from 1a | What the cell counts | What the result would mean |
|---|---|---|---|
| 1 | Some titles still contain HTML codes such as `&#039;` | The cells with a code like `&#039;` or `&amp;`, per text column, and the most frequent codes | If many cells are affected, titles must be decoded (`html.unescape`) before matching or counting words |
| 2 | The 11th may be an automatic day in start dates | The number of start dates (`aired`) for each day of the month, 1 to 31 | If day 11 is far above the other days, it is a default day and those dates are only month-accurate |
| 3 | January 1 may be a default birth day | The share of January 1 among all birth dates, the most frequent month-day values, and the share of January 1 for some birth years | If January 1 is far above 1/365 (0.27%), only the year of those users can be trusted |
| 4 | `aired` is empty when the date is only partial, and `aired_string` still holds it | For the anime with an empty `aired`: what kind of text `aired_string` holds (full date, month and year, year only, empty). The same for the anime with status `Not yet aired` and an empty `aired` (the 147 of notebook 02) | If `aired_string` holds a year or a month, the release year must be read from `aired_string`, and "not yet aired without a start date" does not mean "without any date" |

**Limits.** The cell only describes what is in the two small tables. It cannot tell us why a default day exists, only that the counts look like one.


In [4]:
# NEW: four checks on the guesses of part 1a (they only read the two small tables)
import ast, html

# 1. HTML codes in the text columns
ENTITY = r"&#?\w+;"
print("1. Cells with an HTML code (for example &#039; or &amp;)")
for label, df in (("anime", anime), ("users", users)):
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        s = df[col].dropna().astype(str)
        hit = s[s.str.contains(ENTITY, regex=True)]
        if len(hit):
            codes = hit.str.findall(ENTITY).explode().value_counts().head(5).to_dict()
            print(f"   {label}.{col}: {len(hit):,} of {len(s):,} cells ({len(hit) / len(s):.2%}) | codes: {codes}")
            print(f"      example: {hit.iloc[0][:60]!r} -> {html.unescape(hit.iloc[0])[:60]!r}")

# 2. Day of the month of the start date in `aired`
def aired_from(t):
    try:
        return pd.to_datetime(ast.literal_eval(t).get("from"), utc=True, errors="coerce")
    except Exception:
        return pd.NaT

start = pd.to_datetime(anime["aired"].map(aired_from), utc=True)
day = start.dt.day.dropna().astype(int)
per_day = day.value_counts().reindex(range(1, 32), fill_value=0)
print()
print(f"2. Day of the month of the start date: {len(day):,} start dates, mean per day {len(day) / 31:.0f}, median day count {per_day.median():.0f}")
print("   count per day:", per_day.to_dict())
print("   top 3 days:", per_day.sort_values(ascending=False).head(3).to_dict())
d11 = start[start.dt.day == 11]
print("   start dates on the 11th, most frequent year-months:", d11.dt.strftime("%Y-%m").value_counts().head(5).to_dict())

# 3. January 1 among the birth dates
bd = pd.to_datetime(users["birth_date"], errors="coerce").dropna()
month_day = bd.dt.strftime("%m-%d")
share_jan1 = (month_day == "01-01").mean()
print()
print(f"3. Birth dates: {len(bd):,} parsed. January 1: {(month_day == '01-01').sum():,} users = {share_jan1:.2%} (uniform days would give {1 / 365:.2%})")
print("   most frequent month-day values:", month_day.value_counts().head(5).to_dict())
by_year = bd[bd.dt.year.between(1985, 2000)]
jan1_by_year = (by_year.dt.strftime("%m-%d") == "01-01").groupby(by_year.dt.year).mean().round(3)
print("   share of January 1 per birth year, 1985-2000:", jan1_by_year.to_dict())

# 4. `aired` against `aired_string`
def kind(t):
    if pd.isna(t) or str(t).strip().lower() in ("", "not available"): return "empty"
    if re.search(r"[A-Z][a-z]{2} \d{1,2}, \d{4}", t): return "full date"
    if re.search(r"[A-Z][a-z]{2}, \d{4}", t): return "month and year"
    if re.search(r"\b\d{4}\b", t): return "year only"
    return "other"

string_kind = anime["aired_string"].map(kind)
no_start = start.isna()
print()
print(f"4. `aired` has no start date for {int(no_start.sum()):,} anime. What `aired_string` holds for them:")
print("  ", string_kind[no_start].value_counts().to_dict())
print("   other texts (first 8):", anime.loc[no_start & (string_kind == "other"), "aired_string"].value_counts().head(8).to_dict())
print("   `aired` has a start date but `aired_string` is empty:", int((~no_start & (string_kind == "empty")).sum()))
unaired = no_start & (anime["status"] == "Not yet aired")
print(f"   status `Not yet aired` and no start date in `aired`: {int(unaired.sum()):,}. What `aired_string` holds:")
print("  ", string_kind[unaired].value_counts().to_dict())
print("   most frequent texts:", anime.loc[unaired, "aired_string"].value_counts().head(6).to_dict())


1. Cells with an HTML code (for example &#039; or &amp;)
   anime.title: 202 of 14,478 cells (1.40%) | codes: {'&#039;': 213}
      example: 'Dae Jang Geum: Jang Geum&#039;s Dream' -> "Dae Jang Geum: Jang Geum's Dream"
   anime.title_english: 409 of 5,724 cells (7.15%) | codes: {'&#039;': 426}
      example: 'Sekai Ichi Hatsukoi - World&#039;s Greatest First Love 2' -> "Sekai Ichi Hatsukoi - World's Greatest First Love 2"
   anime.title_japanese: 76 of 14,443 cells (0.53%) | codes: {'&#039;': 79}
      example: '電脳戦隊ヴギィ&#039;ズ★エンジェル外伝 進め! スーパー★エンジェルス!' -> "電脳戦隊ヴギィ'ズ★エンジェル外伝 進め! スーパー★エンジェルス!"
   anime.title_synonyms: 739 of 8,937 cells (8.27%) | codes: {'&#039;': 898}
      example: 'Sekai-ichi Hatsukoi 2, Sekai&#039;ichi Hatsukoi 2' -> "Sekai-ichi Hatsukoi 2, Sekai'ichi Hatsukoi 2"
   anime.background: 316 of 1,057 cells (29.90%) | codes: {'&#039;': 378, '&rsquo;': 26, '&rdquo;': 11, '&mdash;': 11, '&ndash;': 8}
      example: 'Bleach adapts the first 54 volumes of Tite Kubo&#039;s man

### Observations for the four checks

**How to read it.** Each of the four blocks of output tests one guess of part 1a. Below, every guess gets a verdict and the numbers behind it.

**1. HTML codes: widespread, and almost all of them are `&#039;` (an apostrophe: ').**

| Column | Cells with a code | Share of the filled cells |
|---|---|---|
| `anime.background` | 316 | 29.9% |
| `anime.title_synonyms` | 739 | 8.3% |
| `anime.title_english` | 409 | 7.1% |
| `anime.ending_theme` | 393 | 2.7% |
| `anime.opening_theme` | 303 | 2.1% |
| `anime.producer` | 168 | 2.0% |
| `anime.studio` | 130 | 1.5% |
| `anime.related` | 212 | 1.5% |
| `anime.title` | 202 | 1.4% |
| `anime.title_japanese` | 76 | 0.5% |
| `users.location` | 40 | 0.03% |

- The code `&#039;` is the main one (for example `Brain&#039;s Base` for `Brain's Base`). `&amp;` (the character `&`) appears in `producer` (112) and `studio` (40). `background` also has `&rsquo;`, `&rdquo;`, `&mdash;` and `&ndash;`, `related` has `&gt;`, and `users.location` has letters with accents (`&eacute;`, `&atilde;`, `&ouml;`, ...).
- **Verdict:** the guess is right and larger than we thought: 11 text columns are affected, and the same company or title can exist in two spellings (`Brain's Base` and `Brain&#039;s Base`). Every text column must be decoded with `html.unescape` before any matching, grouping by studio or producer, or word count.

**2. A default day 11: not a general rule, but it exists for partial dates (explained in part 7).**

| Day of the month | Start dates |
|---|---|
| 25 (highest) | 808 |
| 21 | 678 |
| 5 | 623 |
| 1 | 612 |
| **11** | **457** |
| mean over the 31 days | 412 |
| median day | 359 |
| 31 (lowest, not every month has it) | 159 |

- Over all 12,787 start dates, the 11th is only 1.1 times the mean. It is not a peak of its own.
- The 11th is high **only in some months of 2018**: of its 457 dates, 107 are in `2018-05` (60), `2018-07` (27) and `2018-10` (20); every other year-month has at most 7. (Part 1a counted 57 for `2018-05-11` from the exact text; the count by date is 60.) Part 7 (cell A5b) explains them: they are dates that were filled in because the text of `aired_string` was only partly written (for example `Jul, 2018` stored with a made-up day).
- Two other peaks are unexplained: the 25th (808) and the 21st (678). They may be real release habits or another default; this was not tested.
- **Verdict:** day 11 is not a default day in general. But a date written only as a month and a year is stored with a filled-in day (9 to 12) and sometimes a filled-in year (part 7, A5b), so such dates are not reliable at the day level. The peaks on the 21st and 25th stay open.

**3. January 1 as a default birth day: supported.**
- **7,353 of the 168,741 parsed birth dates are January 1: 4.36%.** If the days were spread evenly, the share would be 0.27% (about 462 users). So January 1 is **16 times too frequent**, and about 6,900 users have a January 1 that is probably not a real birthday.
- The share is not the same for every birth year: **6.5% for users born in 1985 and 1.9% for users born in 2000**, falling almost every year (1985: 6.5%, 1990: 4.7%, 1995: 3.5%, 2000: 1.9%). Older users are more often listed with January 1.
- The next most frequent month-days are the 1st of July (619), April (613), October (595) and May (589), about 1.3 times the 462 expected. This small rise on the 1st of the month was not tested further.
- **Verdict:** the guess fits the data. We cannot tell which of the 7,353 are real birthdays (about 462 are expected to be). So the **birth year** is the only part to trust: use the age in years, do not use the month or the day of birth. Not checked: whether the 842 users born in 1930 (part 6) are also January 1.

**4. `aired` against `aired_string`: confirmed for the anime without a start date.**

| `aired_string` for the 1,691 anime without a start date in `aired` | Anime |
|---|---|
| A year only (for example `2005`) | 1,114 |
| A month and a year (for example `Jul, 2018 to ?`) | 354 |
| Empty (`Not available`) | 223 |
| A full date or any other text | 0 |

- So `aired` is empty when the date is only partial: **1,468 anime (86.8% of the 1,691, 10.1% of all 14,478) have a year or a month in `aired_string`** and nothing in `aired`. For the other 223 there is no date at all. Also, no anime has a start date in `aired` and an empty `aired_string`.
- **Consequence:** the release year must be read from `aired_string` (a year, or a month and a year). Reading `aired` alone loses 1,468 anime. Not checked here: whether an anime with a year-only `aired_string` can still have a start date in `aired` (we tested only one direction). Part 7 (A5b) shows that some partial texts, such as `Oct, 2007 to Oct, 2008`, **do** have a start date in `aired`, but it is filled in and wrong.
- **The 147 anime with status `Not yet aired` and no start date (notebook 02):** `aired_string` is `Not available` for **113** of them (no date at all), a year for 19 and a month and a year for 15. The most frequent texts of the 34 partial ones are `2018` (14), `2019` (5), `Mar, 2018` (5), `Oct, 2018` (2) and `Nov, 2018` (2). So the guess is right for 34 of the 147: they have a month or a year, mostly 2018 or 2019, which fits "not yet aired" at the time of the crawl (May 2018). One exception to look at: `Mar, 2018` (5 anime) is before the crawl, so the label may be out of date or the premiere was delayed. We did not check this. The conclusion of notebook 02 is not weakened.

**What this changes.**
- Summary rows 17 and 20 to 22 are updated with the counts above.
- For step 4: decode HTML codes in all text columns, read the release year from `aired_string`, treat January 1 as an unknown day of birth, and do not trust the day or the year of a start date whose text is partial (part 7).
- **Checks still open:** the peaks on the 21st and 25th, the 1st of the month in birth dates, January 1 for the 842 users born in 1930, and the reverse direction of check 4.


In [5]:
NA_TOKENS = {"null", "NULL", "NA", "N/A", "n/a", "NaN", "nan", "None", "#N/A", "#NA",
             "-NaN", "-nan", "<NA>", "1.#IND", "1.#QNAN", "-1.#IND", "-1.#QNAN", "#N/A N/A"}

for file_name in ["AnimeList.csv", "UserList.csv"]:
    text = pd.read_csv(paths[file_name], dtype=str, keep_default_na=False)
    hits = {col: int(text[col].isin(NA_TOKENS).sum()) for col in text.columns}
    hits = {col: count for col, count in hits.items() if count}
    print(file_name, "-> columns with hidden NA tokens:", hits or "none")
    for col in hits:
        print("   ", col, text.loc[text[col].isin(NA_TOKENS), col].value_counts().head(5).to_dict())
    del text

# Safe load: only the empty string is missing
na_kwargs = dict(keep_default_na=False, na_values=[""])
anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)
print("\nanime:", anime.shape, "| users:", users.shape)
print("username NaN:", users.username.isna().sum(),
      "| username unique:", users.username.is_unique,
      "| distinct usernames:", f"{users.username.nunique():,}", "(description: 302,675)")
assert users.username.is_unique

AnimeList.csv -> columns with hidden NA tokens: {'rating': 544}
    rating {'None': 544}
UserList.csv -> columns with hidden NA tokens: {'username': 1, 'location': 26}
    username {'null': 1}
    location {'N/A': 15, 'n/a': 7, 'NA': 3, 'None': 1}

anime: (14478, 31) | users: (302675, 17)
username NaN: 0 | username unique: True | distinct usernames: 302,675 (description: 302,675)


### Observations for part 1

**What we did.** Every cell of the two small tables was read as raw text, and we counted the cells that pandas would have turned into a missing value (`null`, `NA`, `N/A`, `None`, ...). Then both tables were loaded again with the safe load.

| File | Column | Hidden words | Cells |
|---|---|---|---|
| `AnimeList.csv` | `rating` | `None` | 544 |
| `UserList.csv` | `username` | `null` | 1 |
| `UserList.csv` | `location` | `N/A` 15, `n/a` 7, `NA` 3, `None` 1 | 26 |

No other cell of either table holds one of these words.

- **`username`: the fix is confirmed a second time.** With the safe load there are 0 missing usernames and 302,675 distinct usernames, as in the description, and the `assert` on uniqueness passed.
- **`rating = None` (544 anime).** With the default load pandas would have turned these into missing values without telling us. They are a word that means "no age rating". Whether to count them as missing is a cleaning decision for step 4.
- **`location` (26 cells).** Users typed these words themselves. `N/A`, `n/a` and `None` mean "no answer". `NA` (3 cells) could be the same non-answer or an abbreviation, and we cannot tell. These 26 cells are only the part that pandas hides: the first look (part 1a) found **309 placeholder cells in `location`** (0.2% of the 156,799 filled cells). The safe load keeps all of them as text.


## 2. Anime table

Columns that matter for the candidate questions: `type`, `source`, `episodes`, `duration`, `aired_string`, `genre`, `status`, `rating` (age rating), `related` (sequels, prequels), and the audience columns `score`, `scored_by`, `rank`, `popularity`, `members`, `favorites`.

For a model that predicts the score **before an anime airs**, the audience columns (`score`, `scored_by`, `rank`, `popularity`, `members`, `favorites`) are only known after airing, so they must be excluded as leakage.

In [6]:
def missing_overview(df):
    return pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
    }).sort_values("missing_pct", ascending=False)


print(anime.dtypes.value_counts().to_dict())
print(missing_overview(anime).head(12))

# Placeholder values that are not NaN but also carry no information
PLACEHOLDERS = {"", "Unknown", "Not available", "None", "?", "0000-00-00"}
text_cols = anime.select_dtypes(exclude="number").columns
placeholder_counts = pd.Series({c: int(anime[c].isin(PLACEHOLDERS).sum()) for c in text_cols})
print("\nplaceholder values per column:")
print(placeholder_counts[placeholder_counts > 0].sort_values(ascending=False))

print("\nduplicated anime_id:", anime.anime_id.duplicated().sum())

{dtype('O'): 22, dtype('int64'): 6, dtype('float64'): 2, dtype('bool'): 1}
                missing_count  missing_pct
background              13421        92.70
licensor                11105        76.70
premiered               10382        71.71
broadcast               10207        70.50
title_english            8754        60.46
producer                 6190        42.75
studio                   5934        40.99
title_synonyms           5541        38.27
rank                     1574        10.87
image_url                  96         0.66
genre                      64         0.44
title_japanese             35         0.24

placeholder values per column:
source          4210
broadcast       2241
rating           544
duration         405
aired_string     223
type              30
dtype: int64

duplicated anime_id: 0


In [7]:
# Numeric columns: are the ranges plausible?
num_cols = ["score", "scored_by", "rank", "popularity", "members", "favorites", "episodes"]
print(anime[num_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T.round(2))

# How reliable is the published score?
print("\nscore == 0:", (anime.score == 0).sum(),
      "| scored_by == 0:", (anime.scored_by == 0).sum(),
      "| both:", ((anime.score == 0) & (anime.scored_by == 0)).sum())
for k in (10, 50, 100, 1000):
    share = (anime.scored_by < k).mean()
    print(f"anime with scored_by < {k:>4}: {(anime.scored_by < k).sum():>6,} ({share:.1%})")
print("episodes == 0:", (anime.episodes == 0).sum())

              count      mean       std  min      1%      25%      50%  \
score       14478.0      6.14      1.46  0.0    0.00     5.55     6.37   
scored_by   14478.0  11460.03  43105.19  0.0    0.00    46.00   501.00   
rank        12904.0   6439.07   3720.23  0.0  120.06  3216.25  6441.50   
popularity  14478.0   7220.26   4170.08  0.0  136.77  3611.50  7225.50   
members     14478.0  22966.40  74981.36  0.0   29.00   245.00  1679.50   
favorites   14478.0    311.65   2615.55  0.0    0.00     0.00     2.00   
episodes    14478.0     11.31     43.44  0.0    0.00     1.00     1.00   

                 75%        99%        max  
score           7.06       8.46       10.0  
scored_by    3941.50  194311.41  1009477.0  
rank         9664.00   12742.97    12919.0  
popularity  10827.75   14300.23    14487.0  
members     10379.00  361159.94  1456378.0  
favorites      23.00    6780.55   106895.0  
episodes       12.00     104.00     1818.0  

score == 0: 354 | scored_by == 0: 308 | both: 

In [8]:
# Categorical columns
for col in ["type", "source", "status", "rating"]:
    print(f"\n{col}: {anime[col].nunique(dropna=False)} distinct values")
    print(anime[col].value_counts(dropna=False).head(12))


type: 7 distinct values
type
TV         4271
OVA        3612
Movie      2624
Special    1948
ONA        1144
Music       849
Unknown      30
Name: count, dtype: int64

source: 16 distinct values
source
Unknown         4210
Original        3368
Manga           3120
Visual novel     888
Game             607
Light novel      563
Other            408
Novel            359
Music            314
4-koma manga     224
Web manga        150
Picture book      96
Name: count, dtype: int64

status: 3 distinct values
status
Finished Airing     13791
Currently Airing      351
Not yet aired         336
Name: count, dtype: int64

rating: 7 distinct values
rating
PG-13 - Teens 13 or older         5020
G - All Ages                      4541
PG - Children                     1279
Rx - Hentai                       1219
R - 17+ (violence & profanity)     997
R+ - Mild Nudity                   878
None                               544
Name: count, dtype: int64


In [9]:
# Release year from aired_string (the first 4-digit number is the start year)
aired_year = pd.to_numeric(anime["aired_string"].str.extract(r"(\d{4})", expand=False), errors="coerce")
print("aired_string without a year:", aired_year.isna().sum(), f"({aired_year.isna().mean():.1%})")
print(aired_year.describe().round(1))
print(anime.loc[aired_year.isna(), "aired_string"].value_counts().head(5))
print("examples:", anime["aired_string"].dropna().head(5).tolist())

# Episode length in minutes from the text in `duration`
print("\n", anime["duration"].value_counts().head(8))
hours = pd.to_numeric(anime["duration"].str.extract(r"(\d+)\s*hr")[0], errors="coerce").fillna(0)
minutes = pd.to_numeric(anime["duration"].str.extract(r"(\d+)\s*min")[0], errors="coerce").fillna(0)
has_time = anime["duration"].str.contains(r"\d+\s*(?:hr|min)", na=False)
duration_min = (hours * 60 + minutes).where(has_time)
print("duration not parsable:", (~has_time).sum(), f"({(~has_time).mean():.1%})")
print(duration_min.describe(percentiles=[.01, .5, .99]).round(1))

aired_string without a year: 223 (1.5%)
count    14255.0
mean      2003.1
std         14.9
min       1917.0
25%       1997.0
50%       2008.0
75%       2014.0
max       2019.0
Name: aired_string, dtype: float64
aired_string
Not available    223
Name: count, dtype: int64
examples: ['Jan 13, 2012 to Mar 30, 2012', 'Apr 2, 2007 to Oct 1, 2007', 'Oct 4, 2008 to Sep 25, 2009', 'Aug 16, 2002 to May 23, 2003', 'Oct 6, 2012 to Mar 30, 2013']

 duration
24 min. per ep.    1513
25 min. per ep.     808
23 min. per ep.     797
30 min. per ep.     749
2 min.              476
Unknown             405
3 min.              371
4 min.              336
Name: count, dtype: int64
duration not parsable: 718 (5.0%)
count    13760.0
mean        26.8
std         25.5
min          1.0
1%           1.0
50%         24.0
99%        115.0
max        231.0
Name: 0, dtype: float64


In [10]:
# genre holds several values in one cell
genres = anime["genre"].dropna().str.split(", ")
print("anime without genre:", anime["genre"].isna().sum())
print("genres per anime:", genres.str.len().describe().round(2).to_dict())
print("distinct genres:", genres.explode().nunique())
print(genres.explode().value_counts().head(10))

# related: how are sequels stored, and how many anime have one?
print("\nexample:", anime["related"].dropna().head(2).tolist())
for relation in ["Sequel", "Prequel", "Side story", "Spin-off"]:
    share = anime["related"].str.contains(f"'{relation}'", na=False).mean()
    print(f"anime with a '{relation}' entry: {share:.1%}")

anime without genre: 64
genres per anime: {'count': 14414.0, 'mean': 2.91, 'std': 1.65, 'min': 1.0, '25%': 2.0, '50%': 3.0, '75%': 4.0, 'max': 13.0}
distinct genres: 43
genre
Comedy           5251
Action           3253
Fantasy          2703
Adventure        2582
Drama            2294
Sci-Fi           2272
Kids             2003
Shounen          1771
Romance          1634
Slice of Life    1532
Name: count, dtype: int64

example: ["{'Adaptation': [{'mal_id': 17207, 'type': 'manga', 'url': 'https://myanimelist.net/manga/17207/Inu_x_Boku_SS', 'title': 'Inu x Boku SS'}], 'Sequel': [{'mal_id': 13403, 'type': 'anime', 'url': 'https://myanimelist.net/anime/13403/Inu_x_Boku_SS_Special', 'title': 'Inu x Boku SS Special'}]}", "{'Adaptation': [{'mal_id': 759, 'type': 'manga', 'url': 'https://myanimelist.net/manga/759/Seto_no_Hanayome', 'title': 'Seto no Hanayome'}], 'Sequel': [{'mal_id': 4163, 'type': 'anime', 'url': 'https://myanimelist.net/anime/4163/Seto_no_Hanayome_OVA', 'title': 'Seto no Hanay

### Observations for part 2 (anime table)

**Missing values.** The columns that are mostly empty are the same as in step 2: `background` 92.7%, `licensor` 76.7%, `premiered` 71.7%, `broadcast` 70.5%, `title_english` 60.5%, `producer` 42.8%, `studio` 41.0%, `title_synonyms` 38.3%. `rank` misses 1,574 values (10.9%), and `genre` (64) and `title_japanese` (35) are almost complete. There is no repeated `anime_id`.

**Placeholders: words that mean "no value" but that `isna()` does not see.**

| Column | Cells that hold a placeholder word | Share of 14,478 anime |
|---|---|---|
| `source` | 4,210 | 29.1% |
| `broadcast` | 2,241 | 15.5% |
| `rating` | 544 | 3.8% |
| `duration` | 405 | 2.8% |
| `aired_string` | 223 | 1.5% |
| `type` | 30 | 0.2% |

All 2,241 `broadcast` placeholders are the word `Unknown` (part 6). For `broadcast` the placeholders add to the empty cells: 10,207 empty + 2,241 placeholders = 12,448 anime (86.0%) without a usable broadcast. A column that is empty or a placeholder for 86% of the rows is not a feature.

**Numbers: the ranges, and why the mean is not the typical value.**

| Column | Median | Mean | Maximum |
|---|---|---|---|
| `scored_by` | 501 | 11,460 | 1,009,477 |
| `members` | 1,680 | 22,966 | 1,456,378 |
| `favorites` | 2 | 312 | 106,895 |
| `episodes` | 1 | 11.3 | 1,818 |

In every row the mean is far above the median, so a few very popular anime pull the mean up. Half of all anime have at most 1 episode (zeros included), and half have at most 501 raters. For these columns we use the median or a log scale, not the mean.

**Score 0 means "no score".** 354 anime have `score = 0`. Of these, 308 also have `scored_by = 0`, so nobody rated them, and 0 is a placeholder. The other 46 have `score = 0` although `scored_by` is above 0. Part 6 shows that the ten with the most raters are all `Not yet aired`, with at most 15 raters: a score is not published before an anime airs. Either way, `score = 0` must become a missing value before any mean.

**Values to check.**
- `rank` and `popularity` each have **9 anime with the value 0**, although a rank starts at 1 (counted in part 6). A 0 is a placeholder.
- `episodes = 0` for **504 anime** (3.5%) means "unknown number of episodes", not "no episode". `episodes` is 1 for half of the anime (movies, specials, music videos), and the maximum 1,818 is a long-running series, which is plausible.

**How reliable is the published `score`?** A mean of a few votes moves a lot with one vote. Five votes of 8 give a mean of 8, and one more vote of 1 gives 6.83. The number of anime with few raters:

| Raters (`scored_by`) | Anime below this number | Share |
|---|---|---|
| 10 | 1,276 | 8.8% |
| 50 | 3,718 | 25.7% |
| 100 | 4,789 | 33.1% |
| 1,000 | 8,489 | 58.6% |

One anime in three has fewer than 100 raters, so its `score` is noisy. Keeping only the anime with at least 100 raters leaves 9,689 of 14,478. The team must choose a threshold for any question about quality, and report how many anime it removes.

**Categories.**
- `type` has 7 values: TV 4,271, OVA 3,612, Movie 2,624, Special 1,948, ONA 1,144, Music 849 and `Unknown` 30.
- `source` has 16 values, and `Unknown` (4,210) is the most frequent one.
- `rating` (age rating) has 6 real values and `None` (544). `Rx - Hentai` has 1,219 anime (8.4%), which matters for any question about audiences or scores.
- `status` is the same as in step 2: Finished Airing 13,791, Currently Airing 351, Not yet aired 336.

**Release year and episode length.**
- `aired_string` gives a start year for 14,255 anime (98.5%). The other 223 hold `Not available`. The years run from 1917 to 2019 with a median of 2008 (a quarter of the anime start after 2014, a quarter before 1997). No year is impossible. The year 2019 belongs to anime that had not aired at the crawl.
- `duration` could be turned into minutes for 13,760 anime. 718 (5.0%) cannot be parsed by our code: 405 are `Unknown` (really missing) and 313 are lengths written in seconds, such as `30 sec.` (part 6), which our code did not read. Only the 405 are missing. The median is 24 minutes, and 99% are at most 115 minutes (movies hold the total length).

**Genres and `related`.**
- `genre` holds several values in one cell. The 14,414 anime that have a genre carry 2.9 genres on average (1 to 13), with 43 different genres. The most frequent are Comedy (5,251 anime), Action (3,253) and Fantasy (2,703). To count by genre we must split the cell into one row per genre.
- `related` is parsable (the text of a dictionary). About 16.8% of the anime (around 2,430) have a `Sequel` entry and the same share have a `Prequel` entry, which fits the mirrored links we measured in step 5a. 7.3% have a `Side story` and 2.5% a `Spin-off`. A question about sequels therefore rests on about 2,400 anime with a sequel.

**What this means for a model that predicts the score before an anime airs.** The audience columns (`score`, `scored_by`, `rank`, `popularity`, `members`, `favorites`) are only known after airing, so they are excluded (leakage). The columns left are `type`, `source`, `episodes`, `duration`, `genre`, `rating`, `studio`, `producer`, the start year and `related`. Their problems are `source = Unknown` for 29%, `studio` and `producer` missing for about 41% and 43%, `episodes = 0` for 3.5%, and `rating = None` for 3.8%.


## 3. User table

In [11]:
count_cols = ["user_watching", "user_completed", "user_onhold", "user_dropped", "user_plantowatch"]
stat_cols = ["user_days_spent_watching", "stats_mean_score", "stats_rewatched", "stats_episodes"]

print(users.dtypes.value_counts().to_dict())
print(missing_overview(users).head(8))
print("access_rank non-missing:", users["access_rank"].notna().sum())

print("\ngender:")
print(users["gender"].value_counts(dropna=False))
print("\ndistinct locations:", users["location"].nunique(),
      "| missing:", users["location"].isna().sum())
print(users["location"].value_counts().head(10))

{dtype('O'): 6, dtype('int64'): 6, dtype('float64'): 5}
                  missing_count  missing_pct
access_rank              302675       100.00
location                 145876        48.20
birth_date               133926        44.25
gender                    84875        28.04
stats_episodes              129         0.04
join_date                   129         0.04
stats_rewatched             129         0.04
stats_mean_score            129         0.04
access_rank non-missing: 0

gender:
gender
Male          135181
NaN            84875
Female         81374
Non-Binary      1245
Name: count, dtype: int64

distinct locations: 55283 | missing: 145876
location
Poland         2425
Germany        1947
Canada         1821
California     1510
Brazil         1459
France         1135
Philippines    1077
Australia      1032
Sweden         1013
Finland         898
Name: count, dtype: int64


In [12]:
# Dates: placeholders and implausible values
for col in ["birth_date", "join_date", "last_online"]:
    parsed = pd.to_datetime(users[col], errors="coerce")
    print(f"{col:12s} unparsable/missing: {parsed.isna().sum():>7,}",
          "| '0000-00-00':", int((users[col] == "0000-00-00").sum()),
          "| earliest:", parsed.min(), "| latest:", parsed.max())
    users[col + "_dt"] = parsed

age_2018 = (pd.Timestamp("2018-01-01") - users["birth_date_dt"]).dt.days / 365.25
print("\nage at 2018-01-01:")
print(age_2018.describe(percentiles=[.01, .05, .5, .95, .99]).round(1))
print("age < 8:", int((age_2018 < 8).sum()), "| age > 80:", int((age_2018 > 80).sum()))
print("last_online before 2000:", int((users["last_online_dt"].dt.year < 2000).sum()),
      "| joined after last_online:", int((users["join_date_dt"] > users["last_online_dt"]).sum()))

# Aggregates reported per user
print(users[stat_cols].describe(percentiles=[.01, .5, .99, .999]).T.round(2))
print("stats_mean_score == 0:", int((users["stats_mean_score"] == 0).sum()))
print("users declaring no anime at all:", int((users[count_cols].sum(axis=1) == 0).sum()))

birth_date   unparsable/missing: 133,934 | '0000-00-00': 0 | earliest: 1800-01-01 00:00:00 | latest: 2137-01-01 00:00:00
join_date    unparsable/missing:     129 | '0000-00-00': 0 | earliest: 2004-11-05 00:00:00 | latest: 2018-05-24 00:00:00
last_online  unparsable/missing:     129 | '0000-00-00': 0 | earliest: 1900-01-01 00:13:00 | latest: 2018-05-25 12:53:00

age at 2018-01-01:
count    168741.0
mean         24.9
std           7.0
min        -119.0
1%           14.7
5%           17.6
50%          24.2
95%          33.0
99%          44.4
max         218.0
Name: birth_date_dt, dtype: float64
age < 8: 769 | age > 80: 955
last_online before 2000: 89283 | joined after last_online: 89283
                             count     mean        std  min   1%      50%  \
user_days_spent_watching  302675.0    69.08    4741.02  0.0  0.0    39.58   
stats_mean_score          302546.0     7.12       2.57  0.0  0.0     7.80   
stats_rewatched           302546.0    10.38      44.24  0.0  0.0     0.00   

### Observations for part 3 (user table)

**Missing values.** `access_rank` is empty for all 302,675 users, so we drop it. `location` is missing for 48.2%, `birth_date` for 44.3% and `gender` for 28.0%. `join_date`, `last_online` and the three `stats_*` columns miss 129 values each (the same 129 users, see part 4 of the notebook 02).

**Gender.** Among the 217,800 users who gave a gender: Male 135,181 (62.1%), Female 81,374 (37.4%) and Non-Binary 1,245 (0.6%). The labels are clean. Non-Binary is a small group, so a comparison involving it has wide uncertainty.

**Location.** 55,283 different texts. The most frequent are Poland (2,425), Germany (1,947), Canada (1,821), California (1,510) and Brazil (1,459). Countries and a US state sit side by side, so a country analysis needs a grouping step first.

**Dates.**

| Column | Earliest | Latest | Meaning |
|---|---|---|---|
| `join_date` | 2004-11-05 | 2018-05-24 | Plausible (MyAnimeList started in 2004) |
| `last_online` | 1900-01-01 | 2018-05-25 12:53 | The year 1900 is impossible, see below |
| `birth_date` | 1800-01-01 | 2137-01-01 | Impossible years at both ends |

- **`last_online` holds the year 1900 for many users.** 89,283 users (29.5%) have a `last_online` before the year 2000, and the same 89,283 users also "joined after they were last online". Part 6 shows that **all 89,283 are in exactly the year 1900** (for example 1900-05-13 02:47), while the other users spread over 2007 to 2018. So this is one default year, not a real date. The year was probably missing from the source, so the month, day and time may be right and the year is not. The true year is unknown (the examples we saw are from March to May). The author says the cleaned version fixes these values. For 29.5% of the users `last_online` cannot be used as a date.
- **`birth_date`.** 8 texts cannot be parsed (133,934 unparsable against 133,926 empty). They are fake dates with the years 1, 1337, 6969 and 9001 (part 6). 168,741 users have a usable date. The age at 2018-01-01 is 24.9 on average and 24.2 at the median. The middle 98% of the users are between 14.7 and 44.4 years old, so the bulk is plausible. But **769 users are under 8 and 955 are over 80** (1,724 users, 1.0% of those with a date), with extremes of -119 and 218 years. These are wrong or joke dates. Part 6 adds that the 769 young ones are spread over the birth years 2010 to 2018, while **842 of the 955 old ones were born in 1930**, which looks like a default value (a hypothesis: the first year of a drop-down list). Any age analysis must cut the age range.
- **The crawl date, a third view.** The latest `last_online` is 2018-05-25 12:53, the latest `join_date` is 2018-05-24, and earlier we found 2018-05-22 for `my_last_updated` and 2018-05-25 for the anime start dates. All four agree: the data was read around 22 to 25 May 2018.

**The numbers MyAnimeList reports for each user.**

| Column | Median | 99th percentile | Maximum | Problem |
|---|---|---|---|---|
| `user_days_spent_watching` | 39.6 | 304 | **2,601,351** | The maximum is about 7,100 years |
| `stats_mean_score` | 7.80 | 10.00 | 10.00 | 30,922 users have 0 |
| `stats_rewatched` | 0 | 149 | 9,404 | Unclear meaning (step 5) |
| `stats_episodes` | 2,384 | 19,893 | **243,794,900** | The maximum is impossible |

- **`stats_mean_score = 0` for 30,922 users (10.2% of the 302,546 with a value).** As for `my_score`, 0 here means "no score given", not a mean of 0, so it must become a missing value.
- **Impossible maxima.** The mean of `stats_episodes` (4,846) is far above its median (2,384) because of values like 243,794,900 episodes, which is about 11,000 years of non-stop watching at 24 minutes per episode. The value 10,989,789 that we met in step 5a was the largest among the users compared there; over all users the maximum is 243,794,900. These values need a cap or removal. Part 6 counts the users above several caps (204 users above 100,000 episodes, 20 above 1,000,000, and 42 above 3,650 days) and shows that the largest values come from absurd episode numbers.
- **19,628 users declare no anime at all.** Part 4 explains how this relates to the 19,630 users without list rows.


## 4. Do the per-user counts agree with the list table?

`UserList` reports how many anime each user has in each status. `UserAnimeList` holds the actual rows. If both come from the same crawl, the counts should match. This also settles the 8 users in step 1 who declare anime but have no rows.

In [13]:
if RUN_LISTS:
    lists = pd.read_csv(
        paths["UserAnimeList.csv"], usecols=["username", "my_status"],
        dtype={"username": "category", "my_status": "int8"}, **na_kwargs,
    )
    print(f"list rows: {len(lists):,} | username NaN: {lists.username.isna().sum()}")

    per_status = lists.groupby(["username", "my_status"], observed=True).size().unstack(fill_value=0)
    print("usernames in lists but not in users:", len(set(per_status.index) - set(users.username)))

    status_to_col = {1: "user_watching", 2: "user_completed", 3: "user_onhold",
                     4: "user_dropped", 6: "user_plantowatch"}
    actual = per_status.reindex(columns=list(status_to_col)).fillna(0).astype(int)
    actual.columns = [c + "_actual" for c in status_to_col.values()]
    actual.index = actual.index.astype(object)
    declared = users.set_index("username")[count_cols].add_suffix("_declared")
    merged = declared.join(actual, how="left").fillna(0)
    print("merge rows:", len(merged), "| users:", len(users))

    for col in count_cols:
        diff = merged[col + "_actual"] - merged[col + "_declared"]
        print(f"{col:18s} exact match: {(diff == 0).mean():6.1%} | median diff: {diff.median():.0f}"
              f" | mean abs diff: {diff.abs().mean():.2f}")

    declared_total = merged[[c + "_declared" for c in count_cols]].sum(axis=1)
    actual_total = merged[[c + "_actual" for c in count_cols]].sum(axis=1)
    print("\ndeclare 0 but have rows:", int(((declared_total == 0) & (actual_total > 0)).sum()),
          "| declare > 0 but have no rows:", int(((declared_total > 0) & (actual_total == 0)).sum()))
else:
    print("Skipped (RUN_LISTS = False)")

list rows: 80,076,112 | username NaN: 0
usernames in lists but not in users: 0
merge rows: 302675 | users: 302675
user_watching      exact match:  98.2% | median diff: 0 | mean abs diff: 0.03
user_completed     exact match:  98.7% | median diff: 0 | mean abs diff: 0.07
user_onhold        exact match:  99.9% | median diff: 0 | mean abs diff: 0.01
user_dropped       exact match:  99.9% | median diff: 0 | mean abs diff: 0.01
user_plantowatch   exact match:  99.4% | median diff: 0 | mean abs diff: 0.04

declare 0 but have rows: 6 | declare > 0 but have no rows: 8


### Observations for part 4 (declared counts against the list table)

**What we did.** `UserList` says how many anime each user has per status. We counted the real rows of `UserAnimeList` per user and status, and compared the two. Every user of the user table is in the comparison (302,675 rows), and 0 usernames of the list table are missing from the user table, so the join on `username` has no orphan on either side.

| Status | Users whose declared and actual counts are equal | Mean absolute difference |
|---|---|---|
| watching | 98.2% | 0.03 |
| completed | 98.7% | 0.07 |
| on hold | 99.9% | 0.01 |
| dropped | 99.9% | 0.01 |
| plan to watch | 99.4% | 0.04 |

The median difference is 0 for every status.

- **The two tables agree for at least 98.2% of the users in each status.** The profile counts and the list rows come from the same crawl and describe the same lists. The list table is almost complete for the users it holds.
- **The disagreement is small and is largest for `watching` (1.8% of the users, about 5,400) and `completed` (1.3%, about 3,900).** These are the statuses that change most often, so a plausible reason is that a user changed the list between the moment the profile was read and the moment the list was read. This is a hypothesis. The output does not show the direction of the differences.
- **8 users declare anime but have no rows.** This is the same group as in step 1, now confirmed with a second method.
- **6 users declare 0 anime but have rows.** This is the opposite case and is new. The reason is unknown, and 6 users are too few to matter.
- **Direction (part 6).** For `completed` the list has fewer rows than the profile in 91% of the mismatches, and for `plan to watch` it has more rows in 79%. This fits a list that was read earlier than the profile. The 6 users who declare 0 anime but have rows have long lists (294 to 2,057 rows) and are not explained.
- **Consequence.** `UserList` counts can be used as a summary of the list, with a small margin. We do not need to treat them as unreliable.


## 5. The users with no row in the list table: is the list table missing them?

In step 1 we found **19,630 users** who exist in `UserList` but own no row in `UserAnimeList`. For 19,622 of them the profile declares 0 anime, so an empty list is the natural explanation. But a declared count is only one column. If these users also have **other activity values** (days spent watching, episodes watched, a mean score), their list may exist on MyAnimeList and simply be missing from our list table.

| What we see among these users | What it means |
|---|---|
| `user_days_spent_watching`, `stats_episodes` and `stats_mean_score` are all 0 | The list is really empty. Nothing is missing |
| Many have positive values there | They watched and rated anime, so their list is **missing from the list table** (not crawled, private, or lost) |
| Their profile fields (gender, birth date, location) are filled as often as for other users | They are normal active profiles. If those fields are mostly empty, they are probably inactive accounts |

In [14]:
if RUN_LISTS:
    users["has_rows"] = users["username"].isin(actual.index)
    no_rows = users[~users["has_rows"]]
    with_rows = users[users["has_rows"]]
    declared_size = no_rows[count_cols].sum(axis=1)
    print(f"users without a row in the list table: {len(no_rows):,} | with rows: {len(with_rows):,}")
    print(f"  declared list size 0: {int((declared_size == 0).sum()):,} | declared list size > 0: {int((declared_size > 0).sum()):,}")

    print("\nactivity values of the users without rows (an empty account would show 0 everywhere):")
    for col in ["user_days_spent_watching", "stats_episodes", "stats_rewatched", "stats_mean_score"]:
        s = no_rows[col]
        print(f"  {col:26s} missing: {int(s.isna().sum()):>6,} | zero: {int((s == 0).sum()):>6,} | positive: {int((s > 0).sum()):>6,}"
              f" | max: {s.max():,.1f}")
    active = ((no_rows["user_days_spent_watching"] > 0) | (no_rows["stats_episodes"] > 0) | (no_rows["stats_mean_score"] > 0))
    print(f"\nusers without rows that show activity in at least one of the three columns: {int(active.sum()):,}")

    print("\nprofile fields filled (share of users), compared with the users that have rows:")
    fields = ["gender", "birth_date", "location", "join_date", "last_online"]
    print(pd.DataFrame({"no_rows": no_rows[fields].notna().mean(),
                        "with_rows": with_rows[fields].notna().mean()}).round(3).to_string())

    print("\nusers without rows that declare anime (the unexplained group):")
    cols_show = ["username"] + count_cols + ["user_days_spent_watching", "stats_episodes", "stats_mean_score"]
    print(no_rows.loc[declared_size > 0, cols_show].sort_values("user_completed", ascending=False).head(15).to_string(index=False))
else:
    print("Skipped (RUN_LISTS = False)")

users without a row in the list table: 19,630 | with rows: 283,045
  declared list size 0: 19,622 | declared list size > 0: 8

activity values of the users without rows (an empty account would show 0 everywhere):
  user_days_spent_watching   missing:      0 | zero: 19,624 | positive:      6 | max: 260.7
  stats_episodes             missing:      2 | zero: 19,593 | positive:     35 | max: 30,300.0
  stats_rewatched            missing:      2 | zero: 19,624 | positive:      4 | max: 112.0
  stats_mean_score           missing:      2 | zero: 19,593 | positive:     35 | max: 10.0

users without rows that show activity in at least one of the three columns: 47

profile fields filled (share of users), compared with the users that have rows:
             no_rows  with_rows
gender         0.311      0.748
birth_date     0.263      0.578
location       0.221      0.539
join_date      1.000      1.000
last_online    1.000      1.000

users without rows that declare anime (the unexplained group):


### Observations for part 5 (the users with no row in the list table)

**The question.** In step 1 we found 19,630 users with no row in the list table (283,045 users have rows). Is the list table missing their lists, or are they empty accounts? We look at the other values of their profile: if they were active, they would show days spent watching, episodes or a mean score above 0.

| Value for the 19,630 users without rows | Zero | Above 0 | Missing | Largest |
|---|---|---|---|---|
| `user_days_spent_watching` | 19,624 | 6 | 0 | 260.7 |
| `stats_episodes` | 19,593 | 35 | 2 | 30,300 |
| `stats_mean_score` | 19,593 | 35 | 2 | 10 |
| `stats_rewatched` | 19,624 | 4 | 2 | 112 |

- **The users are empty accounts, with a few exceptions.** 19,622 of them declare a list size of 0, and **only 47 (0.24%) show any activity** in at least one of the three columns. For the other 19,583 users (99.76%) the three columns are 0 (or missing for 2 users), which is what an account without a list looks like. Nothing is missing for them.
- **Their profiles are less filled in.** Share of users that gave the field:

| Field | Users without rows | Users with rows |
|---|---|---|
| `gender` | 31.1% | 74.8% |
| `birth_date` | 26.3% | 57.8% |
| `location` | 22.1% | 53.9% |
| `join_date`, `last_online` | 100% | 100% |

  Users without rows fill in gender, birth date and location about 2.2 to 2.4 times less often. This fits accounts that were created and never used. (The 2 users with a missing `join_date` are the empty-profile users of notebook 02.)
- **The 8 users who declare anime are the open group.** Six of them show days spent watching above 0 (from 0.22 to 260.7 days; `Useless_Goddess` declares 1,043 completed anime), but all 8 show `stats_episodes = 0` and `stats_mean_score = 0`. So their profile has the counts but not the episode and score summaries. One of them is named `deactivated-180120`, but part 6 shows that the name explains nothing: 10 usernames start with `deactivated`, and 6 of them still own list rows. So we cannot explain any of the 8 (a private list, or a list that was not crawled).
- **Decision for step 4 (the team decides):** the 19,583 empty accounts can be dropped from any analysis of lists. An inner merge of the two tables does this by itself, and it removes no row of the list table. The 47 users with activity and the 8 with a declared list are reported as users whose list may be missing.


## 6. Clarity checks (NEW, added after the first run)

Parts 1 to 5 ran once. Some sentences of their observations rest on numbers that the first run did not print, and some hypotheses were never tested. This part prints them. It changes nothing in the data, and it reuses the objects built above (`anime`, `users`, `age_2018`, `merged`, ...), so run the notebook from the top.

| Check | What it settles |
|---|---|
| A. Anime values | How many `rank` and `popularity` are 0? Which words fill `broadcast`? What are the 313 `duration` texts that cannot be parsed? Which anime have `score = 0` but raters? |
| B. User dates and ages | Is `last_online` before 2000 always the year 1900 (our hypothesis)? Which `birth_date` texts cannot be parsed? Which birth years give the impossible ages? |
| C. Extremes | How many users lie above several caps of `stats_episodes` and `user_days_spent_watching`, and what do the largest ones look like? This helps to choose a cap in step 4. |
| D. `deactivated` usernames | How many usernames start with `deactivated`, and do they own list rows? (One of the 8 users who declare anime without rows is `deactivated-180120`.) |
| E. Declared against actual counts | In which direction do the counts differ, and who are the 6 users that declare 0 anime but have rows? |


In [15]:
# NEW A: anime values that the first run did not print
print("rank == 0:", int((anime["rank"] == 0).sum()), "| rank missing:", int(anime["rank"].isna().sum()),
      "| popularity == 0:", int((anime["popularity"] == 0).sum()))

print("\nbroadcast: the words that count as placeholders")
print(anime.loc[anime["broadcast"].isin(PLACEHOLDERS), "broadcast"].value_counts().to_string())

has_time_text = anime["duration"].str.contains(r"\d+\s*(?:hr|min)", na=False)
print("\nduration: the texts that cannot be parsed into minutes")
print(anime.loc[~has_time_text, "duration"].value_counts(dropna=False).head(10).to_string())

odd_score = (anime["score"] == 0) & (anime["scored_by"] > 0)
print("\nscore == 0 although scored_by > 0:", int(odd_score.sum()))
print(anime.loc[odd_score, ["anime_id", "title", "type", "status", "scored_by", "members"]]
      .sort_values("scored_by", ascending=False).head(10).to_string(index=False))


rank == 0: 9 | rank missing: 1574 | popularity == 0: 9

broadcast: the words that count as placeholders
broadcast
Unknown    2241

duration: the texts that cannot be parsed into minutes
duration
Unknown            405
30 sec.             54
30 sec. per ep.     42
15 sec.             30
15 sec. per ep.     17
40 sec. per ep.      8
45 sec.              7
39 sec. per ep.      6
52 sec.              5
16 sec.              5

score == 0 although scored_by > 0: 46
 anime_id                                                   title    type        status  scored_by  members
    37799                               Tokyo Ghoul:re 2nd Season      TV Not yet aired         15    28258
    37675                                            Overlord III      TV Not yet aired         12    60107
    35760                             Shingeki no Kyojin Season 3      TV Not yet aired         11   170892
    34134                                         One Punch Man 2      TV Not yet aired         10   234

In [16]:
# NEW B: user dates and ages
last_year = users["last_online_dt"].dt.year
print("last_online before 2000, by year:", last_year[last_year < 2000].value_counts().sort_index().to_dict())
print("last_online year counts from 2000 on:", last_year[last_year >= 2000].value_counts().sort_index().to_dict())

unparsable = users["birth_date"].notna() & users["birth_date_dt"].isna()
print("\nbirth_date texts that cannot be parsed:", users.loc[unparsable, "birth_date"].tolist())

birth_year = users["birth_date_dt"].dt.year
print("birth years of the users under 8 at 2018-01-01:", birth_year[age_2018 < 8].value_counts().head(8).to_dict())
print("birth years of the users over 80 at 2018-01-01:", birth_year[age_2018 > 80].value_counts().head(8).to_dict())


last_online before 2000, by year: {1900.0: 89283}
last_online year counts from 2000 on: {2007.0: 638, 2008.0: 5799, 2009.0: 11576, 2010.0: 14067, 2011.0: 12666, 2012.0: 12653, 2013.0: 17328, 2014.0: 15868, 2015.0: 18420, 2016.0: 24231, 2017.0: 40663, 2018.0: 39354}

birth_date texts that cannot be parsed: ['1-06-27', '1-04-22', '1-01-01', '1337-01-01', '1-01-01', '1337-01-01', '9001-12-31', '6969-01-01']
birth years of the users under 8 at 2018-01-01: {2012.0: 91, 2013.0: 87, 2011.0: 86, 2018.0: 82, 2014.0: 82, 2015.0: 79, 2010.0: 73, 2016.0: 71}
birth years of the users over 80 at 2018-01-01: {1930.0: 842, 1931.0: 41, 1934.0: 15, 1933.0: 14, 1937.0: 13, 1932.0: 13, 1935.0: 8, 1936.0: 8}


In [17]:
# NEW C: how many users lie above several caps, and what do the largest look like?
caps = {"stats_episodes": [10_000, 50_000, 100_000, 1_000_000],
        "user_days_spent_watching": [365, 1_000, 3_650, 36_500]}
for col, values in caps.items():
    print(col, {f"> {v:,}": int((users[col] > v).sum()) for v in values})

print()
show = ["username", "stats_episodes", "user_days_spent_watching", "user_completed", "user_plantowatch"]
print(users.nlargest(8, "stats_episodes")[show].to_string(index=False))


stats_episodes {'> 10,000': 19176, '> 50,000': 602, '> 100,000': 204, '> 1,000,000': 20}
user_days_spent_watching {'> 365': 1673, '> 1,000': 192, '> 3,650': 42, '> 36,500': 3}

       username  stats_episodes  user_days_spent_watching  user_completed  user_plantowatch
        warrose     243794881.0                2601351.02            2034                 7
      mmmapples      18218986.0                  23640.42               0                 0
        Kaiyuso      16791524.0                   2278.30             380               479
darkraigiratina      16776960.0                   8555.96               0                 2
 Kitayama_Naoki      10989789.0                   1376.93            2051               443
      sasuke423       5433345.0                    707.53               0                 1
          Fovez       3547267.0                  30197.12             802                 3
      Confucius       3287576.0                 165535.90             719              

In [18]:
# NEW D: usernames that start with "deactivated"
deactivated = users[users["username"].str.lower().str.startswith("deactivated")]
print("usernames starting with 'deactivated':", len(deactivated))
if "has_rows" in users:
    print("  of them with at least one list row:", int(deactivated["has_rows"].sum()))
print(deactivated[["username", "user_completed", "user_plantowatch", "user_days_spent_watching"]].head(10).to_string(index=False))


usernames starting with 'deactivated': 10
  of them with at least one list row: 6
             username  user_completed  user_plantowatch  user_days_spent_watching
      deactivatedshio             170                28                     74.16
     deactivated13041               0                 0                      0.00
     deactivated20325               0                 0                      0.00
       Deactivated453              13                26                      8.49
deactivated_yorozuuya              70                 0                     22.98
     deactivated30325               0                 0                      0.00
       deactivated000               7                 6                      2.73
   deactivated-180120             262                88                    108.67
       deactivated106             715               422                    205.91
          Deactivated              56                22                     47.28


In [19]:
# NEW E: declared against actual counts, direction of the differences
if RUN_LISTS:
    for col in count_cols:
        d = merged[col + "_actual"] - merged[col + "_declared"]
        print(f"{col:18s} actual > declared: {int((d > 0).sum()):>6,} | actual < declared: {int((d < 0).sum()):>6,}"
              f" | largest gap: {int(d.abs().max()):,}")

    zero_declared = merged[(declared_total == 0) & (actual_total > 0)]
    print("\nusers that declare 0 anime but have rows:")
    print(zero_declared[[c + "_actual" for c in count_cols]].assign(total_rows=actual_total[zero_declared.index]).to_string())
else:
    print("Skipped (RUN_LISTS = False)")


user_watching      actual > declared:  2,954 | actual < declared:  2,510 | largest gap: 255
user_completed     actual > declared:    329 | actual < declared:  3,482 | largest gap: 1,043
user_onhold        actual > declared:     92 | actual < declared:    171 | largest gap: 215
user_dropped       actual > declared:     93 | actual < declared:    171 | largest gap: 208
user_plantowatch   actual > declared:  1,488 | actual < declared:    391 | largest gap: 1,432

users that declare 0 anime but have rows:
                  user_watching_actual  user_completed_actual  user_onhold_actual  user_dropped_actual  user_plantowatch_actual  total_rows
username                                                                                                                                   
ker0chan                          51.0                  574.0                 0.0                  0.0                   1432.0      2057.0
Sora-Senpai                        0.0                   93.0            

### Observations for the clarity checks

**A. Anime values.**
- **`rank` and `popularity` equal to 0: 9 anime each.** A rank starts at 1, so 0 is a placeholder. It is a small number, next to the 1,574 anime whose `rank` is empty. Whether the 9 anime are the same for both columns was not checked.
- **`broadcast`: one placeholder word.** All 2,241 placeholders are the word `Unknown`, so a single rule is enough.
- **`duration`: the 313 texts we could not parse are real values, not missing ones.** They are lengths written in **seconds**: `30 sec.` (54 anime), `30 sec. per ep.` (42), `15 sec.` (30), `15 sec. per ep.` (17), `40 sec. per ep.` (8), `45 sec.` (7), and so on. The nine most frequent such texts cover 174 of the 313, and the rest are probably the same kind (not printed). Our code only knew `hr` and `min`. Very short clips and advertisements have such lengths. Only the 405 `Unknown` (2.8%) are really missing, not 718 (5.0%). Rule for step 4: read `sec` too, and divide by 60.
- **`score = 0` although there are raters: 46 anime, and the ten with the most raters are all `Not yet aired`.** They have 4 to 15 raters and large `members` (for example *Tokyo Ghoul:re 2nd Season*, *Overlord III*, *Shingeki no Kyojin Season 3* and *One Punch Man 2*, each with 28,000 to 234,000 members). In part 1 of notebook 02 we counted 46 anime labelled `Not yet aired` that have raters, so these are very probably the same 46 anime (not checked). The reading: **a score is not published before an anime airs**, so `score = 0` there means "no published score", not "bad". This fits the plan to leave the `Not yet aired` anime out of score questions.

**B. User dates and ages.**
- **All 89,283 `last_online` values before 2000 are in exactly the year 1900.** Real dates would spread over several years. The users with a real year go from 638 in 2007 up to 40,663 in 2017 and 39,354 in 2018 (until May). So the year 1900 is one default value, and the year was probably lost, which supports our hypothesis. The month, day and time of these values may still be right, but **the true year is unknown**. The examples we saw are from March to May, and a replacement of 1900 by 2018 would be a guess. For these 29.5% of the users `last_online` cannot be used as a date.
- **The 8 `birth_date` texts that pandas cannot parse are fake dates:** the years 1, 1337, 6969 and 9001 (pandas holds only the years 1677 to 2262).
- **Users under 8 (769): the birth years spread from 2010 to 2018**, (the eight most frequent years have 71 to 91 users each, and 82 users were born in 2018, so their age is 0). There is no single default value. These dates are wrong or joke dates.
- **Users over 80 (955): 842 of them (88.2%) were born in 1930.** One year holding almost all of them looks like a **default value**, for example the first year of a drop-down list when a user signs up. This is a hypothesis. Only 113 users over 80 have another birth year.

**C. Extremes, and where to put a cap.**

| Column | Users above the value | |
|---|---|---|
| `stats_episodes` | 10,000: 19,176 / 50,000: 602 / 100,000: 204 / 1,000,000: 20 | |
| `user_days_spent_watching` | 365: 1,673 / 1,000: 192 / 3,650: 42 / 36,500: 3 | |

- **Scale.** At 24 minutes per episode, 1,000,000 episodes are about 46 years of non-stop watching, and 100,000 episodes are about 4.6 years. So the 20 users above 1,000,000 are impossible, and the 204 above 100,000 are very doubtful. 3,650 days is 10 years of non-stop watching, and 42 users are above it.
- **The largest values come from absurd episode numbers, not from real watching.** `warrose` has 243,794,881 episodes and 2,601,351 days, with 2,034 completed anime. `mmmapples` has 18,218,986 episodes and 23,640 days with **no completed and no planned anime**. `sasuke423` has 5,433,345 episodes with 0 completed anime. Two of the largest values lie almost exactly at 2^24 = 16,777,216: `darkraigiratina` has 16,776,960 and `Kaiyuso` 16,791,524. This is the same kind of storage limit that we saw for `my_rewatching_ep` in step 2, so these values are probably a counter that overflowed.
- **The two summaries do not always agree with each other.** `warrose` has 15 minutes of watching time per episode, which is normal. `Kitayama_Naoki` has 10,989,789 episodes but only 1,377 days, which is 11 seconds per episode. So we cannot repair these users by recomputing one column from the other.
- **Proposal for step 4 (the team decides).** Cap `stats_episodes` and `user_days_spent_watching` at their 99.9th percentile (77,358 episodes and 758 days, from part 3), or remove the users above 100,000 episodes (204, which is 0.07%), and report the number.

**D. Usernames that start with `deactivated`: the name alone explains nothing.** There are 10 such usernames, and **6 of them still own list rows** (for example `deactivatedshio` with 170 completed anime). Of the 4 without rows, three declare 0 anime (empty accounts) and one is `deactivated-180120`, which declares 262 completed anime. So the hypothesis "a deactivated account whose list was removed" is not supported: most deactivated accounts keep their rows. **We cannot explain any of the 8 users who declare anime without rows.**

**E. Declared against actual counts: the direction is not random.**

| Status | Actual above declared | Actual below declared |
|---|---|---|
| watching | 2,954 | 2,510 |
| completed | 329 | 3,482 |
| on hold | 92 | 171 |
| dropped | 93 | 171 |
| plan to watch | 1,488 | 391 |

- **For `completed` the list has fewer rows than the profile says in 91% of the mismatches (3,482 of 3,811), and for `plan to watch` it has more rows in 79% (1,488 of 1,879).** Think of users who finish the anime they planned to watch: over time `completed` goes up and `plan to watch` goes down. This pattern is what we see if the **list was read earlier than the profile**: the profile (newer) shows more completed and fewer planned anime than the rows (older). `watching` is balanced (2,954 against 2,510). This is a hypothesis about the order of the two reads, but it explains the three directions with one story. It also explains why the gaps are small for most users.
- **Part of the biggest gaps are the users without rows.** The largest gap for `completed` (1,043) is `Useless_Goddess`, who declares 1,043 completed anime and has no row.
- **The 6 users who declare 0 anime but have rows have long lists.** `ker0chan` has 2,057 rows, `Omiyage` 895, `AliNyman` 565, `Sora-Senpai` 335, `Ahngel` 294 and `4895624815796354` 294, which makes 4,440 rows. Their profile counts are 0 although the lists are not, and the "older list" story above does not explain this. For these 6 users the declared counts cannot be used.


## 7. Checks that parts 1 to 6 did not make: the list table, numeric columns, usernames, duplicates and contradictions (NEW)

**Purpose.** When we read parts 1 to 6 again as a whole, we found places where we looked at one table or one kind of column and not at the others, or where we described a column without testing it. This part runs those checks, so that the audit has no blind spot before the cleaning decisions of step 4. It changes nothing in the data and reuses `anime`, `users`, `paths` and `RUN_LISTS` from the cells above.

**How to run.** Part 7A only reads the two small tables (seconds). Part 7B reads the 80M-row list table once, 3M rows at a time (about 4 minutes); `RUN_LISTS = False` skips it. The pass and its reports are separate cells, so a mistake in a report does not need a second pass.

**What was missing, and why it matters.**

| # | Gap in notebook 03 | Why it matters | Where |
|---|---|---|---|
| 1 | The list table (80M rows, the largest table) was never scanned for placeholders. The first look (1a) skipped it on purpose because of its size. | A placeholder in the largest table affects the most rows. `my_tags` is free text, and the two date columns can hold default or impossible dates | B1 to B3 |
| 2 | Numeric columns were never scanned for placeholder numbers (`-1`, `999`, `9999`). The first look (1a) read text columns only. | A number such as `-1` or `9999` standing for "unknown" would enter every mean and correlation | A1 |
| 3 | The usernames were checked for being unique (exact match), not for **capital letters and spaces**. | The username is the join key between `UserList` and `UserAnimeList`. Two spellings of one name, or a trailing space, break a join without any error | A2, B4 |
| 4 | `location` was only counted by its most frequent values. | If `USA`, `usa` and `Usa ` are three values, every grouping by country is split | A3 |
| 5 | Duplicates were checked for `anime_id` only. | The same anime may exist under two ids, which would double its weight | A4 |
| 6 | No check that two columns of the same row agree (`scored_by` against `members`, end date against start date, a finished anime that starts after the crawl). | A row that contradicts itself is wrong in at least one column | A5, A5b, A5c, B3 |
| 7 | The columns that hold Python text (`aired`, `related`, `opening_theme`, `ending_theme`) were read with `ast.literal_eval` in notebook 02 without counting the failures. | A parse failure silently drops a row from any analysis of these columns | A6 |
| 8 | The dates of the list table were seen as a top-5 list in notebook 02, and never tested for partial dates, impossible dates, dates after the crawl, or a finish date before the start date. `my_last_updated` was only used for its maximum. | A date range decides which rows can be used for a time analysis | B3 |
| 9 | `anime_id` of the list table was never checked against `AnimeList`, and `my_watched_episodes` was never compared with the episode count. | An id without an anime cannot be joined. More episodes watched than exist is impossible | B4, B5 |

**Evidence tags.** Everything printed is a count from the files ([V]). Explanations of *why* a value exists are hypotheses and are labelled so.


In [20]:
# Setup for part 7 (the objects `anime`, `users`, `paths`, `na_kwargs` and `RUN_LISTS` come from the cells above)
from collections import Counter
import ast, html, re

CRAWL_END = pd.Timestamp("2018-05-25", tz="UTC")  # estimated end of the crawl (notebook 02)


### 7A. The two small tables

### A1. Numeric columns: impossible values and placeholder numbers

For every numeric column (the id columns are skipped) the cell prints how many cells are missing, the minimum, the maximum, how many values are negative, the five most frequent values, and how many cells equal a number that is often used as a placeholder (`-1`, `99`, `999`, `9999`, `99999`). A value such as `999` can also be real (for example a count of favorites), so a hit is a reason to look, not a proof.


In [21]:
# A1. Numeric columns: impossible values, most frequent values, placeholder-like numbers
SENTINELS = [-1, 99, 999, 9999, 99999]

def numeric_overview(df, label, n=5):
    print("=====", label)
    for col in df.select_dtypes(include="number").columns:
        if col.endswith("_id"):
            continue
        s = df[col].dropna()
        top = " | ".join(f"{v:,.6g} x{c:,}" for v, c in s.value_counts().head(n).items())
        sentinel = {v: int((s == v).sum()) for v in SENTINELS if (s == v).any()}
        extra = f" | placeholder-like: {sentinel}" if sentinel else ""
        print(f"{col}: missing {int(df[col].isna().sum()):,} | min {s.min():,.6g} | max {s.max():,.6g}"
              f" | negative {int((s < 0).sum()):,} | top: {top}{extra}")

numeric_overview(anime, "anime table")
print()
numeric_overview(users, "user table")


===== anime table
episodes: missing 0 | min 0 | max 1,818 | negative 0 | top: 1 x6,857 | 2 x1,187 | 12 x1,034 | 13 x637 | 3 x558 | placeholder-like: {99: 2}
score: missing 0 | min 0 | max 10 | negative 0 | top: 0 x354 | 6 x120 | 5 x100 | 6.5 x90 | 6.33 x87
scored_by: missing 0 | min 0 | max 1.00948e+06 | negative 0 | top: 0 x308 | 8 x155 | 7 x143 | 5 x140 | 9 x139 | placeholder-like: {99: 11, 999: 2}
rank: missing 1,574 | min 0 | max 12,919 | negative 0 | top: 0 x9 | 6,303 x5 | 3,012 x4 | 12,630 x4 | 4,700 x4 | placeholder-like: {99: 1, 999: 1, 9999: 1}
popularity: missing 0 | min 0 | max 14,487 | negative 0 | top: 0 x9 | 3,252 x4 | 6,104 x4 | 2,870 x4 | 9,537 x4 | placeholder-like: {99: 1, 999: 1}
members: missing 0 | min 0 | max 1.45638e+06 | negative 0 | top: 123 x35 | 87 x35 | 85 x32 | 115 x32 | 81 x30 | placeholder-like: {99: 17}
favorites: missing 0 | min 0 | max 106,895 | negative 0 | top: 0 x5,018 | 1 x1,539 | 2 x817 | 3 x577 | 4 x407 | placeholder-like: {99: 6, 999: 1}

===== 

### A2. Spaces, capital letters and the username (the join key)

The first cell counts, in every text column of both tables, the cells with a space at the start or at the end, and the cells with two or more spaces in a row. The second one tests the usernames: are there names that differ only by capital letters or by a space? On MyAnimeList a profile address does not depend on capital letters, so two such names could be one person; and a join on the exact text would then miss it.


In [22]:
# A2. Spaces in text cells, and usernames that differ only by capital letters
def spacing(df, label):
    print("=====", label)
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        s = df[col].dropna().astype(str)
        edge = int((s != s.str.strip()).sum())
        double = int(s.str.contains(r"\s{2,}", regex=True).sum())
        if edge or double:
            print(f"{col}: space at the start or the end {edge:,} | two or more spaces in a row {double:,}")

spacing(anime, "anime table")
print()
spacing(users, "user table")

name = users["username"]
low = name.str.lower()
clash = low.duplicated(keep=False)
print()
print("usernames:", f"{len(name):,}", "| distinct:", f"{name.nunique():,}", "| distinct after lowercase:", f"{low.nunique():,}")
print("names that differ only by capital letters:", int(clash.sum()), "in", low[clash].nunique(), "groups")
print(users.loc[clash, ["username", "user_id"]].sort_values("username", key=lambda s: s.str.lower()).head(10).to_string(index=False))
print("usernames with a space at the start or the end:", int((name != name.str.strip()).sum()))


===== anime table
title: space at the start or the end 0 | two or more spaces in a row 3
title_japanese: space at the start or the end 4 | two or more spaces in a row 32
title_synonyms: space at the start or the end 0 | two or more spaces in a row 5
background: space at the start or the end 105 | two or more spaces in a row 29
related: space at the start or the end 0 | two or more spaces in a row 2
opening_theme: space at the start or the end 0 | two or more spaces in a row 40
ending_theme: space at the start or the end 0 | two or more spaces in a row 82

===== user table
location: space at the start or the end 4,560 | two or more spaces in a row 406

usernames: 302,675 | distinct: 302,675 | distinct after lowercase: 302,675
names that differ only by capital letters: 0 in 0 groups
Empty DataFrame
Columns: [username, user_id]
Index: []
usernames with a space at the start or the end: 0


### A3. `location`: how many spellings hide behind the same place?

`location` is free text. The cell compares the number of distinct raw texts with the number after we remove capital letters and extra spaces, and prints the places that have the most spellings. This tells how much grouping work step 4 has before any analysis by country or city.


In [23]:
# A3. location: raw texts against texts without capital letters and extra spaces
loc = users["location"].dropna()
norm = loc.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
print("filled locations:", f"{len(loc):,}", "| distinct raw texts:", f"{loc.nunique():,}", "| distinct after normalising:", f"{norm.nunique():,}")
print("most frequent after normalising:", norm.value_counts().head(10).to_dict())

spellings = loc.groupby(norm).nunique().sort_values(ascending=False)
print("\nplaces written in the most different ways:")
for key in spellings.head(5).index:
    forms = loc[norm == key].value_counts().head(6).to_dict()
    print(f"  {key!r}: {int(spellings[key])} spellings, for example {forms}")


filled locations: 156,799 | distinct raw texts: 55,283 | distinct after normalising: 49,616
most frequent after normalising: {'poland': 2544, 'germany': 2004, 'canada': 1946, 'california': 1679, 'brazil': 1517, 'philippines': 1281, 'france': 1204, 'australia': 1101, 'sweden': 1080, 'singapore': 952}

places written in the most different ways:
  'saudi arabia': 12 spellings, for example {'Saudi Arabia': 82, 'saudi arabia': 19, 'Saudi Arabia ': 12, ' Saudi Arabia': 5, 'saudi arabia ': 3, 'Saudi arabia': 2}
  'new york': 12 spellings, for example {'New York': 661, 'new york': 45, 'New York ': 24, 'New york': 15, ' New York': 4, 'NEW YORK': 3}
  'los angeles, california': 11 spellings, for example {'Los Angeles, California': 325, 'Los Angeles, California ': 16, 'los angeles, california': 7, 'Los angeles, California': 3, 'los angeles, california ': 1, 'Los Angeles, California     ': 1}
  'somewhere over the rainbow': 11 spellings, for example {'Somewhere over the rainbow': 31, 'somewhere ov

### A4. Duplicates beyond `anime_id`

Notebook 03 only checked that `anime_id` is unique. Here we look for the same title under several ids, and for rows that are identical in every column except `anime_id` (the same anime entered twice).


In [24]:
# A4. Duplicate anime that have different ids
dup_title = anime[anime["title"].duplicated(keep=False)]
print("titles used by more than one anime_id:", dup_title["title"].nunique(), "| rows:", len(dup_title))
print(dup_title.sort_values("title")[["anime_id", "title", "type", "status", "scored_by"]].head(10).to_string(index=False))

same_but_id = anime.drop(columns=["anime_id"]).duplicated(keep=False)
print("\nrows identical in every column except anime_id:", int(same_but_id.sum()))

title_clean = anime["title"].str.strip().str.lower()
print("titles that differ only by capital letters or spaces:", int(title_clean.duplicated(keep=False).sum() - anime["title"].duplicated(keep=False).sum()))


titles used by more than one anime_id: 1 | rows: 2
 anime_id       title type           status  scored_by
    35102 Hinamatsuri  OVA  Finished Airing          8
    36296 Hinamatsuri   TV Currently Airing      12470

rows identical in every column except anime_id: 0
titles that differ only by capital letters or spaces: 0


### A5. Do the columns of one row agree with each other?

A row that contradicts itself is wrong in at least one column. The cell counts: a `score` outside 0 to 10; more raters than members (`scored_by` > `members`); more favorites than members; an end date before the start date in `aired`; a **finished** anime whose start date lies after the crawl; and a **currently airing** anime whose end date lies before the crawl (the crawl ended about 2018-05-25, see notebook 02).


In [25]:
# A5. Contradictions inside one row of the anime table
def aired_part(t, key):
    try:
        return pd.to_datetime(ast.literal_eval(t).get(key), utc=True, errors="coerce")
    except Exception:
        return pd.NaT

air_start = pd.to_datetime(anime["aired"].map(lambda t: aired_part(t, "from")), utc=True)
air_end = pd.to_datetime(anime["aired"].map(lambda t: aired_part(t, "to")), utc=True)

print("score outside 0 to 10:", int(((anime["score"] < 0) | (anime["score"] > 10)).sum()))
print("scored_by > members:", int((anime["scored_by"] > anime["members"]).sum()),
      "| favorites > members:", int((anime["favorites"] > anime["members"]).sum()))
print("end date before start date:", int((air_end < air_start).sum()))
finished_late = (anime["status"] == "Finished Airing") & (air_start > CRAWL_END)
airing_over = (anime["status"] == "Currently Airing") & (air_end < CRAWL_END)
print("Finished Airing with a start date after the crawl:", int(finished_late.sum()))
print("Currently Airing with an end date before the crawl:", int(airing_over.sum()))
cols = ["anime_id", "title", "status", "aired_string"]
print(anime.loc[finished_late, cols].head(5).to_string(index=False))
print(anime.loc[airing_over, cols].head(5).to_string(index=False))


score outside 0 to 10: 0
scored_by > members: 0 | favorites > members: 1
end date before start date: 32
Finished Airing with a start date after the crawl: 16
Currently Airing with an end date before the crawl: 6
 anime_id                    title          status              aired_string
    21277            Yawaraka Atom Finished Airing    Oct, 2007 to Oct, 2008
    33756               Hen Xi You Finished Airing Jun, 2016 to Nov 28, 2016
    13819         Kamiusagi Rope 2 Finished Airing    Nov, 2010 to Dec, 2011
    19695               Clover 4/3 Finished Airing    Aug, 2000 to Dec, 2000
    28733 Ginga Tetsudou 999 (ONA) Finished Airing    Sep, 2002 to Jul, 2004
 anime_id             title           status                 aired_string
    36623        SNS Police Currently Airing Mar 24, 2018 to May 19, 2018
    35857 Lupin III: Part 5 Currently Airing          Apr 4, 2018 to 2018
    36902  Mahou Shoujo Ore Currently Airing          Apr 2, 2018 to 2018
    34572      Black Clover Cu

### A5b. The raw `aired` text of the rows that contradict themselves (NEW)

A5 found rows that contradict themselves, but it printed only the `aired_string`. For the first five "Finished Airing" anime, `aired_string` shows an old date (for example `Oct, 2007 to Oct, 2008`), yet A5 says their start date in `aired` lies after the crawl. So `aired` and `aired_string` seem to disagree for them. This cell prints the raw `aired` text of every group, and tests one guess: **when the end of `aired_string` is only a year or a month, `aired` stores the first day of that period**, which would put the end before the start (for example `Apr 4, 2018 to 2018` would end on 2018-01-01).


In [26]:
# A5b. The raw `aired` text of the anime that contradict themselves, and the guess about partial end dates
def end_kind(t):
    parts = str(t).split(" to ")
    if len(parts) < 2: return "no end text"
    e = parts[1].strip()
    if e == "?": return "still open"
    if re.search(r"[A-Z][a-z]{2} \d{1,2}, \d{4}", e): return "full date"
    if re.search(r"[A-Z][a-z]{2}, \d{4}", e): return "month and year"
    if re.search(r"^\d{4}$", e): return "year only"
    return "other"

groups = {
    "end date before start date": air_end < air_start,
    "Finished Airing, start date after the crawl": finished_late,
    "Currently Airing, end date before the crawl": airing_over,
    "favorites > members": anime["favorites"] > anime["members"],
}
for label, mask in groups.items():
    print(f"--- {label}: {int(mask.sum())} anime")
    print(anime.loc[mask, ["anime_id", "title", "status", "aired_string", "aired"]].head(40).to_string(index=False, max_colwidth=75))

has_end = air_end.notna()
kinds = anime["aired_string"].map(end_kind)
before_start = groups["end date before start date"]
print()
print("how the end of `aired_string` is written, for the anime that have an end date in `aired`:")
print("  all:", kinds[has_end].value_counts().to_dict())
print("  end before start:", kinds[before_start].value_counts().to_dict())
print(f"end date on the 1st of a month: all {(air_end[has_end].dt.day == 1).mean():.1%} | end before start {(air_end[before_start].dt.day == 1).mean():.1%}")


--- end date before start date: 32 anime
 anime_id                                                   title          status                 aired_string                                      aired
    33600                                          Menghuan Xiyou Finished Airing  Dec 29, 2014 to Feb 6, 2015 {'from': '2015-12-29', 'to': '2015-02-06'}
     6822                                  Ukkari Pénélope (2009) Finished Airing  Jun 29, 2009 to Jan 2, 2010 {'from': '2009-06-29', 'to': '2009-01-02'}
    33126                   Kurogane no Linebarrels Picture Drama Finished Airing Dec 14, 2009 to Aug 26, 2009 {'from': '2009-12-14', 'to': '2009-08-26'}
    21195                             Soul Eater: Late Night Show Finished Airing       Apr, 2008 to Mar, 2009 {'from': '2018-04-10', 'to': '2018-03-10'}
    30840                                   Many Things SD Gundam Finished Airing         2011 to Jul 22, 2011 {'from': '2018-05-11', 'to': '2011-07-22'}
    34234                          

### A5c. How many `aired` dates are filled in? (NEW)

A5b showed that a date written only partly in `aired_string` (`Oct, 2007`, `2011`) is stored in `aired` as a complete date, and that the same numbers come back again and again: the year 2018, the month May, and the days 9 to 12. Our hypothesis is that the missing parts were filled with the date of the fetch. This cell tests it on **every** anime, not only on the 54 rows that A5b printed.

For the start and for the end of `aired_string` it says how the text is written (a full date, a month and a year, a year only, `?` for an open end, or absent). For each kind it counts the anime that have a date in `aired` and prints the most frequent days, months and years of these dates. For the full dates it also counts how often the date in `aired` differs from the date written in the text (the year shifted by one, for example).


In [27]:
# A5c. Are the missing parts of a date in aired_string filled in `aired`? (all anime)
def split_aired(t):
    p = str(t).split(" to ")
    return p[0].strip(), (p[1].strip() if len(p) > 1 else None)

def text_kind(e):
    if pd.isna(e): return "absent"
    if e == "?": return "open"
    if re.search(r"^[A-Z][a-z]{2} \d{1,2}, \d{4}$", e): return "full date"
    if re.search(r"^[A-Z][a-z]{2}, \d{4}$", e): return "month and year"
    if re.search(r"^\d{4}$", e): return "year only"
    return "other"

pairs = anime["aired_string"].map(split_aired)
for label, pos, stamp in (("start", 0, air_start), ("end", 1, air_end)):
    texts = pairs.map(lambda p: p[pos])
    text_kinds = texts.map(text_kind)
    print(f"--- {label}: how `aired_string` writes it, and what `aired` holds")
    for k in ["full date", "month and year", "year only", "open", "absent", "other"]:
        in_kind = text_kinds == k
        has_date = in_kind & stamp.notna()
        line = f"  {k:15s} anime: {int(in_kind.sum()):>6,} | with a date in aired: {int(has_date.sum()):>6,}"
        if has_date.any():
            s = stamp[has_date]
            line += (f" | day top 4: {s.dt.day.value_counts().head(4).to_dict()}"
                     f" | month top 3: {s.dt.month.value_counts().head(3).to_dict()}"
                     f" | year top 3: {s.dt.year.value_counts().head(3).to_dict()}")
        print(line)
    is_full = text_kinds == "full date"
    written = pd.to_datetime(texts.where(is_full), format="%b %d, %Y", errors="coerce")
    same = stamp.dt.tz_localize(None).dt.normalize() == written
    compared = is_full & stamp.notna() & written.notna()
    print(f"  full dates in the text that also have a date in aired: {int(compared.sum()):,} | aired differs from the text: {int((compared & ~same).sum()):,}")
    print(anime.loc[compared & ~same, ["anime_id", "aired_string", "aired"]].head(8).to_string(index=False, max_colwidth=70))
    print()


--- start: how `aired_string` writes it, and what `aired` holds
  full date       anime: 12,560 | with a date in aired: 12,560 | day top 4: {25: 799, 21: 678, 5: 623, 1: 612} | month top 3: {4: 1959, 10: 1716, 7: 1477} | year top 3: {2016: 794, 2017: 782, 2014: 755}
  month and year  anime:    476 | with a date in aired:    122 | day top 4: {11: 81, 10: 15, 12: 10, 9: 10} | month top 3: {7: 44, 10: 34, 4: 13} | year top 3: {2018: 106, 1992: 3, 1999: 3}
  year only       anime:  1,219 | with a date in aired:    105 | day top 4: {11: 71, 10: 17, 12: 7, 9: 6} | month top 3: {5: 105} | year top 3: {2018: 85, 1992: 3, 1986: 2}
  open            anime:      0 | with a date in aired:      0
  absent          anime:      0 | with a date in aired:      0
  other           anime:    223 | with a date in aired:      0
  full dates in the text that also have a date in aired: 12,560 | aired differs from the text: 52
 anime_id                aired_string                                      aired
  

### A6. The columns that hold Python text: how many cells cannot be read?

`aired`, `related`, `opening_theme` and `ending_theme` are texts written like a Python dictionary or list. Notebook 02 read them with `ast.literal_eval`, and a cell that fails was silently turned into a missing value. Here the failures are counted.


In [28]:
# A6. Python-literal columns: how many cells fail to parse?
for col in ["aired", "related", "opening_theme", "ending_theme"]:
    s = anime[col].dropna()
    bad = []
    for v in s:
        try:
            ast.literal_eval(v)
        except Exception:
            bad.append(v)
    example = bad[0][:80] if bad else "-"
    print(f"{col:14s} cells: {len(s):>6,} | cannot be parsed: {len(bad):>5,} | example: {example!r}")


aired          cells: 14,478 | cannot be parsed:     0 | example: '-'
related        cells: 14,478 | cannot be parsed:     0 | example: '-'
opening_theme  cells: 14,478 | cannot be parsed:     0 | example: '-'
ending_theme   cells: 14,478 | cannot be parsed:     0 | example: '-'


### Observations for part 7A

**How to read it.** Each block below answers one check. "Cleared" means that we looked for the problem and did not find it.

**A1. Numeric columns: no placeholder number (cleared).**
- No numeric column has a negative value.
- The "placeholder-like" hits (`99`, `999`, `9999`) are false alarms. They are ordinary counts: 17 anime have exactly 99 members and 891 users have exactly 99 completed anime. Every integer appears in a count column, so `99` shows up by itself.
- The most frequent values show only the zeros that we already knew: `score = 0` for 354 anime, `rank = 0` and `popularity = 0` for 9 anime each, `favorites = 0` for 5,018 anime (34.7%), `stats_mean_score = 0` for 30,922 users.
- `access_rank` is missing for all 302,675 users. `stats_mean_score`, `stats_rewatched` and `stats_episodes` are missing for 129 users, the same number as the empty profiles of notebook 02 (we did not check that they are the same users).
- The largest values were already seen in notebook 03: `user_days_spent_watching` up to 2,601,350 days (about 7,000 years) and `stats_episodes` up to 243,795,000 are impossible. `episodes` up to 1,818 is plausible for a very long series.

**A2. Spaces and usernames.**
- **`location` has a space at the start or the end in 4,560 cells (2.9% of the 156,799 filled cells)** and two spaces in a row in 406. The other columns have only a few: `background` 105 and `title_japanese` 4 (edge), `opening_theme` 40, `ending_theme` 82, `title_japanese` 32 and `background` 29 (double spaces). These are minor.
- **Usernames are clean (cleared).** 302,675 distinct names, still 302,675 after lowercase, none with a space at the start or the end. No two names differ only by capital letters, so the join key has no hidden duplicate.

**A3. `location`: a tenth of the spellings disappear when we normalise.**
- 55,283 raw texts become 49,616 after removing capital letters and extra spaces: 5,667 fewer (10.3%).
- Most frequent after normalising: `poland` 2,544, `germany` 2,004, `canada` 1,946, `california` 1,679, `brazil` 1,517, `philippines` 1,281. A state (`california`) stands among countries, so grouping by country needs a mapping that we have to decide in step 4. `los angeles, california` is a city inside a state.
- The places with the most spellings are `saudi arabia` (12), `new york` (12) and `los angeles, california` (11). The differences are capital letters and spaces at the edge (`New York `, ` New York`, `NEW YORK`). Some locations are jokes (`Somewhere over the rainbow`, 11 spellings).
- **Verdict:** strip, lowercase and collapse spaces is one cheap step that removes 10% of the distinct values. The mapping from place to country is a separate decision.

**A4. Duplicate anime (cleared).** One title appears twice: `Hinamatsuri` is `anime_id` 35102 (an OVA, finished) and 36296 (a TV series, currently airing, 12,470 raters). They are two different anime. No row is identical to another except `anime_id`, and no titles differ only by capital letters or spaces. A title is not a key, so never join anime by title.

**A5. Contradictions inside one row: most of them are dates that were filled in.**
- Cleared: `score` outside 0 to 10 (0 anime) and `scored_by` > `members` (0 anime).
- `favorites` > `members`: 1 anime, `Force of Will the Movie` (`anime_id` 33438), not yet aired and without dates. Harmless.
- **End date before start date: 32 anime. "Finished Airing" with a start date after the crawl: 16. "Currently Airing" with an end date before the crawl: 6.** Cell A5b printed the raw `aired` text of all of them (54 rows), and it changes how these three counts must be read.

**What A5b shows.** This is read on the printed rows. A5c, below, counts it on the whole table.
1. **A date that is only partly written in `aired_string` is stored in `aired` as a complete date, with values that were not in the text.** The same numbers come back again and again: the year **2018**, the month **May**, and the days **9, 10, 11 and 12**. Examples: `Oct, 2007 to Oct, 2008` is stored as `from 2018-10-11, to 2018-10-11`; `Dec, 2005 to Nov, 2011` as `from 2018-12-11, to 2018-11-11`; `Apr, 2008 to Mar, 2009` as `from 2018-04-10, to 2018-03-10`; `2011 to Jul 22, 2011` as `from 2018-05-11`; `Jul 5, 1992 to 1992` as `to 1992-05-09`; `Apr 4, 2018 to 2018` as `to 2018-05-10`.
2. **Hypothesis:** the missing year, month and day were filled with the date of the fetch. The anime table would then have been fetched around **9 to 12 May 2018**, earlier than the user tables (22 to 25 May, notebook 02). This is an inference from these rows. The exact filling rule is not worked out: for a missing year the filled year is sometimes 2018 and sometimes the year of the text.
3. **The 16 "Finished Airing" anime with a start after the crawl are not contradictions of the source.** All 16 have a start written as a month and a year (`Oct, 2007`, `Jun, 2016`, ...), and `aired` holds a start in 2018 with a month after May (`2018-10-11`, `2018-06-11`, ...). They are filled-in dates.
4. **The 32 "end before start".** Read by eye on the printed rows (not by code): 21 are filled-in dates (for example `Apr, 2008 to Mar, 2009` stored as `2018-04-10` to `2018-03-10`). The other 11 have a full date in the text. In 7 of them the **text itself is contradictory** (`Mar 10, 1991 to Mar 10, 1990`, `Oct 2, 2016 to Sep 17, 2016`, `Jul 21, 1994 to Jun 21, 1994`), and in 4 the year in `aired` is one year away from the text (`Dec 29, 2014 to Feb 6, 2015` is stored as `from 2015-12-29`, `Jun 29, 2009 to Jan 2, 2010` as `to 2009-01-02`).
5. **The 6 "Currently Airing" with an end before the crawl.** Four are filled-in ends: `Lupin III: Part 5` (`Apr 4, 2018 to 2018`, stored `to 2018-05-10`), `Mahou Shoujo Ore`, `Black Clover` and `Nyanfuru` (`to Mar, 2018`, stored `2018-03-11`). The other two, `SNS Police` and `IDOLiSH7`, end on `May 19, 2018`. If the anime table was fetched on 9 to 12 May, that end lies after the fetch and their status is correct. Against the end of the crawl of the user tables (25 May) the same status would look out of date.

**The guess of A5b was not supported.** A year-only end is not stored as January 1: 4.3% of all end dates fall on the 1st of a month, and 3.1% in the "end before start" group. The days that come back are 9 to 12. The text of the 32 "end before start" is a full date for 18, a month and a year for 9 and a year only for 5. Among the 12,287 anime with an end date in `aired`, the end is written as a year only for 86 and as a month and a year for 60.

**Consequence.** `aired` is reliable only when the text of `aired_string` is a complete date. A filled-in date has a wrong year, month or day, mostly in 2018, so it must not be used for a time trend or for "released before or after the crawl". The release year should come from the text, and only when the text has a year. In step 4 these dates must be flagged. A5c gives the sizes.

**A5c. The filled-in dates counted on the whole table: the hypothesis is supported.**

| How `aired_string` writes it | Anime | With a date in `aired` | Four most frequent days | Month of these dates | Year of these dates |
|---|---|---|---|---|---|
| **Start**, a full date | 12,560 | 12,560 | spread over the month (25th: 799, 21st: 678, 5th: 623, 1st: 612) | spread | spread (real dates) |
| **Start**, a month and a year | 476 | **122** | **11: 81, 10: 15, 12: 10, 9: 10** | 7: 44, 10: 34, 4: 13 | 2018: 106 |
| **Start**, a year only | 1,219 | **105** | **11: 71, 10: 17, 12: 7, 9: 6** | **May for all 105** | 2018: 85 |
| **End**, a full date | 6,467 | 6,467 | spread | spread | spread |
| **End**, a month and a year | 60 | **60** | **11: 44, 9: 9, 10: 5, 12: 1** | 3: 11, 12: 10, 11: 7 | 2018: 50 |
| **End**, a year only | 86 | **86** | **11: 65, 9: 8, 10: 8, 12: 3** | **May for all 86** | 2018: 64 |
| End, `?` (still open) | 500 | 0 | | | |
| End, absent (a single date in the text) | 7,365 | 5,674 | spread | spread | spread |

- **Few dates are filled in, but they are easy to recognise.** Of the 12,787 start dates in `aired`, 12,560 (98.2%) come from a full date in the text and **227 (1.8%) from a partial text**. Of the 12,287 end dates, **146 (1.2%)** come from a partial text. That is 373 dates in all (1.5% of the 25,074 dates).
- **The pattern is the same for starts and ends.** For a year-only text the month is **May in every case** (105 of 105 starts, 86 of 86 ends). The day is **9, 10, 11 or 12** for 360 of the 373 dates (96.5%). The year is **2018** for 305 of them (82%), even when the text says another year (`Oct, 2007` is stored in 2018); a few keep the year of the text (1992, 1999, 1986), and we do not know the exact rule. A fetch on 9 to 12 May 2018 would give exactly this month and these four days, so the hypothesis of the date of the fetch is supported. It is still an inference: we cannot see the crawler.
- **The other partial starts are empty.** 354 month-and-year starts and 1,114 year-only starts have no date in `aired` (476 - 122 and 1,219 - 105). Together they are the 1,468 anime found in part 1b. So a partial text gives either nothing (1,468) or a filled-in date (227); we did not find what decides it. The 1,691 empty `aired` are these 1,468 plus the 223 `Not available`.
- **Full dates can be a year off around New Year.** For 52 of the 12,560 full starts (0.41%) and 19 of the 6,467 full ends (0.29%), the date in `aired` differs from the text. Every printed example is at the turn of the year: 29 to 31 December is stored one year later (`Dec 30, 1996` becomes `1997-12-30`, `Dec 31, 1990` becomes `1991-12-31`), and 1 or 2 January is stored one year earlier (`Jan 1, 1999` becomes `1998-01-01`, `Jan 2, 2010` becomes `2009-01-02`). This is what the **ISO week-numbering year** gives (the week that contains 29 December can belong to the next year, and the week that contains 1 January to the previous one). We checked it on the 16 printed rows: the ISO year of the text date equals the stored year in 16 of 16, with the month and the day unchanged. So it is a likely formatting error in the source (**hypothesis**; the other 55 differing rows were not printed). It explains the 4 "end before start" rows whose year is one off.
- **Consequence for step 4.** The date in `aired` can be used as a date only when the text of `aired_string` is a complete date, and even then the year may be wrong for a date between 29 December and 2 January. For the 373 filled-in dates and the 1,468 empty ones, use the text: the year when the text has one, and nothing else. Notebook 02 (part 4) applies this to the 336 anime labelled `Not yet aired`: 54 of the 60 early start dates are filled-in dates.

**A6. Python text columns (cleared).** All 14,478 cells of `aired`, `related`, `opening_theme` and `ending_theme` can be parsed: 0 failures. The `-` in the output only means that there is no example.


### 7B. The list table (80M rows), read once

The list table was never scanned for placeholders (the first look of notebook 03 skipped it because of its size). It is read once, 3M rows at a time, and each chunk only updates counters. A text column with a few thousand distinct values (the two dates) is counted exactly; `my_tags`, which has far too many distinct values, is counted by pattern and by an approximate top list (the 300 most frequent values of each chunk are merged, so a frequent value is found and its count is a lower bound).

| Cell | What it does |
|---|---|
| B1 | The one pass. Prints only its progress |
| B2 | Report on `my_tags` |
| B3 | Report on the two date columns and on `my_last_updated` |
| B4 | Report on `anime_id` and on the usernames of the list table |
| B5 | Report on `my_watched_episodes` against the episode count |


In [29]:
# B1. One pass over the list table: only counters are updated, the reports are in the next cells
if RUN_LISTS:
    CHUNK = 3_000_000
    DATE_OK = r"^\d{4}-\d{2}-\d{2}$"
    REAL_DATE = r"^(?!0000)\d{4}-(?!00)\d{2}-(?!00)\d{2}$"
    CANDIDATE = r"^\s*(?:|\?+|-+|\.+|_+|0|none|null|nan|n/?a|unknown|unk|tba|tbd|undefined|not available|not specified|no data|0000-00-00)\s*$"
    EPOCH_SITE = pd.Timestamp("2004-11-01", tz="UTC").timestamp()   # MyAnimeList did not exist before
    EPOCH_END = CRAWL_END.timestamp() + 86400                       # end of 2018-05-25

    use = ["username", "anime_id", "my_watched_episodes", "my_status", "my_start_date",
           "my_finish_date", "my_last_updated", "my_tags"]
    dtypes = {"anime_id": "int32", "my_watched_episodes": "float32", "my_status": "int8", "my_last_updated": "float64"}
    episodes_of = anime.set_index("anime_id")["episodes"]
    known_ids = set(anime["anime_id"])

    rows = empty_username = orphan_rows = 0
    names, ids = set(), set()
    date_counts = {"my_start_date": Counter(), "my_finish_date": Counter()}
    both_real = finish_before_start = 0
    tag_filled = tag_html = 0
    tag_placeholder, tag_top = Counter(), Counter()
    upd_min, upd_max = float("inf"), float("-inf")
    upd_nonpositive = upd_before_site = upd_after_crawl = 0
    ep_known = watched_over = completed_rows = completed_zero = completed_less = 0

    reader = pd.read_csv(paths["UserAnimeList.csv"], usecols=use, dtype=dtypes, chunksize=CHUNK, **na_kwargs)
    for k, chunk in enumerate(reader, start=1):
        rows += len(chunk)
        empty_username += int(chunk["username"].isna().sum())
        names.update(chunk["username"].dropna().unique().tolist())
        ids.update(chunk["anime_id"].unique().tolist())
        orphan_rows += int((~chunk["anime_id"].isin(known_ids)).sum())

        for col in date_counts:
            date_counts[col].update(chunk[col].fillna("<empty>").value_counts().to_dict())
        st = chunk["my_start_date"].fillna("")
        fi = chunk["my_finish_date"].fillna("")
        real = st.str.match(REAL_DATE) & fi.str.match(REAL_DATE)
        both_real += int(real.sum())
        finish_before_start += int((real & (fi < st)).sum())

        tags = chunk["my_tags"].dropna()
        tag_filled += len(tags)
        if len(tags):
            tag_html += int(tags.str.contains(r"&#?\w+;", regex=True).sum())
            is_placeholder = tags.str.match(CANDIDATE, flags=re.IGNORECASE)
            tag_placeholder.update(tags[is_placeholder].str.lower().value_counts().to_dict())
            tag_top.update(tags.value_counts().head(300).to_dict())

        upd = chunk["my_last_updated"].dropna()
        if len(upd):
            upd_min, upd_max = min(upd_min, float(upd.min())), max(upd_max, float(upd.max()))
            upd_nonpositive += int((upd <= 0).sum())
            upd_before_site += int((upd < EPOCH_SITE).sum())
            upd_after_crawl += int((upd > EPOCH_END).sum())

        watched = chunk["my_watched_episodes"]
        total_eps = chunk["anime_id"].map(episodes_of)
        has_eps = total_eps > 0
        done = chunk["my_status"] == 2
        ep_known += int(has_eps.sum())
        watched_over += int((has_eps & (watched > total_eps)).sum())
        completed_rows += int(done.sum())
        completed_zero += int((done & (watched == 0)).sum())
        completed_less += int((done & has_eps & (watched < total_eps)).sum())

        if k % 5 == 0:
            print(f"  ... {k} chunks, {rows:,} rows")
    print(f"done: {rows:,} rows in {k} chunks")
else:
    print("Skipped (RUN_LISTS = False)")


  ... 5 chunks, 15,000,000 rows
  ... 10 chunks, 30,000,000 rows
  ... 15 chunks, 45,000,000 rows
  ... 20 chunks, 60,000,000 rows
  ... 25 chunks, 75,000,000 rows
done: 80,076,112 rows in 27 chunks


In [30]:
# B2. my_tags: how many are filled, which are placeholder-like, which are the most frequent
if RUN_LISTS:
    print(f"my_tags filled: {tag_filled:,} of {rows:,} rows ({tag_filled / rows:.1%}) | with an HTML code: {tag_html:,}")
    print("placeholder-like tags:", f"{sum(tag_placeholder.values()):,}", dict(tag_placeholder.most_common(8)))
    print("most frequent tags (approximate, a lower bound):")
    for v, c in tag_top.most_common(10):
        print(f"   {v[:40]!r} {c:,}")
else:
    print("Skipped (RUN_LISTS = False)")


my_tags filled: 4,760,289 of 80,076,112 rows (5.9%) | with an HTML code: 8,758
placeholder-like tags: 53,395 {'undefined': 33194, '-': 6194, 'null': 1957, '.': 1423, '?': 1422, '...': 1162, '---': 1002, '???': 694}
most frequent tags (approximate, a lower bound):
   'undefined' 33,172
   'Hentai' 31,295
   'Sub' 14,664
   'hentai' 14,142
   'Comedy' 13,664
   'anYme' 13,480
   'comedy' 12,967
   '*' 9,237
   'Action' 8,116
   'Dub' 7,825


In [31]:
# B3. The two date columns and my_last_updated
if RUN_LISTS:
    def summarize_dates(counter, label):
        s = pd.Series(counter, dtype="int64")
        total = int(s.sum())
        txt = pd.Series(s.index, index=s.index)
        is_format = txt.str.match(DATE_OK)
        is_empty = txt == "<empty>"
        is_zero = txt == "0000-00-00"
        other = s[~is_format & ~is_empty]
        real_format = is_format & ~is_zero
        parsed = pd.to_datetime(txt.where(real_format), format="%Y-%m-%d", errors="coerce")
        cannot = s[real_format & parsed.isna()]
        good, stamp = s[parsed.notna()], parsed[parsed.notna()]
        before_1990 = int(good[stamp < pd.Timestamp("1990-01-01")].sum())
        after_crawl = int(good[stamp > pd.Timestamp("2018-05-25")].sum())
        print(f"--- {label}: {total:,} rows, {len(s):,} distinct texts")
        print(f"  empty {int(s[is_empty].sum()):,} | 0000-00-00 {int(s[is_zero].sum()):,} | other formats {int(other.sum()):,} {other.head(5).to_dict()}")
        print(f"  right format but not a real date (for example 2010-00-00 or 2018-02-31): {int(cannot.sum()):,} {cannot.head(5).to_dict()}")
        if len(good):
            by_md = good.groupby(good.index.str[5:10]).sum().sort_values(ascending=False)
            by_year = good.groupby(stamp.dt.year).sum().sort_values(ascending=False)
            share_jan1 = by_md.get("01-01", 0) / good.sum()
            print(f"  real dates: {int(good.sum()):,} | earliest {stamp.min().date()} | latest {stamp.max().date()}")
            print(f"  before 1990: {before_1990:,} | after the crawl (2018-05-25): {after_crawl:,}")
            print("  most frequent month-day:", by_md.head(5).to_dict(), f"| share of 01-01: {share_jan1:.2%} (evenly spread: 0.27%)")
            print("  most frequent years:", by_year.head(5).to_dict())

    summarize_dates(date_counts["my_start_date"], "my_start_date")
    print()
    summarize_dates(date_counts["my_finish_date"], "my_finish_date")
    print(f"\nfinish date before start date: {finish_before_start:,} of {both_real:,} rows that have two real dates")

    print("\nmy_last_updated (seconds since 1970):")
    print("  earliest:", pd.to_datetime(upd_min, unit="s"), "| latest:", pd.to_datetime(upd_max, unit="s"))
    print(f"  <= 0: {upd_nonpositive:,} | before November 2004 (MyAnimeList did not exist): {upd_before_site:,} | after 2018-05-25: {upd_after_crawl:,}")
else:
    print("Skipped (RUN_LISTS = False)")


--- my_start_date: 80,076,112 rows, 12,536 distinct texts
  empty 0 | 0000-00-00 69,010,299 | other formats 0 {}
  right format but not a real date (for example 2010-00-00 or 2018-02-31): 425,434 {'2008-00-00': 25904, '2007-00-00': 22607, '2009-00-00': 24032, '2011-00-00': 19978, '2012-00-00': 20127}
  real dates: 10,640,379 | earliest 1693-09-25 | latest 2115-04-22
  before 1990: 1,762 | after the crawl (2018-05-25): 272
  most frequent month-day: {'04-07': 85889, '04-06': 83280, '01-10': 82223, '04-08': 81944, '04-05': 81766} | share of 01-01: 0.42% (evenly spread: 0.27%)
  most frequent years: {2015: 1392932, 2016: 1374371, 2014: 1330945, 2017: 1306770, 2013: 1190143}

--- my_finish_date: 80,076,112 rows, 12,441 distinct texts
  empty 0 | 0000-00-00 66,550,407 | other formats 0 {}
  right format but not a real date (for example 2010-00-00 or 2018-02-31): 351,987 {'2007-00-00': 18480, '2008-00-00': 21644, '2009-00-00': 20172, '2011-00-00': 17397, '2013-00-00': 17083}
  real dates: 13

In [32]:
# B4. anime_id and usernames of the list table
if RUN_LISTS:
    orphan_ids = ids - known_ids
    print(f"anime_id in the list but not in AnimeList: {len(orphan_ids):,} ids, {orphan_rows:,} rows"
          f" | anime without any list row: {len(known_ids - ids):,} of {len(known_ids):,}")

    listed = pd.Series(sorted(names))
    low = listed.str.lower()
    clash = low.duplicated(keep=False)
    print(f"\nusernames in the list table: {len(listed):,} | empty username cells: {empty_username:,}")
    print("distinct after lowercase:", f"{low.nunique():,}", "| names that differ only by capital letters:", int(clash.sum()))
    print("  examples:", listed[clash].head(10).tolist())
    print("  with a space at the start or the end:", int((listed != listed.str.strip()).sum()))

    user_names = set(users["username"])
    lower_users = set(users["username"].str.lower())
    not_found = [n for n in listed if n not in user_names]
    only_case = [n for n in not_found if n.lower() in lower_users]
    print("list usernames not found in UserList (exact text):", len(not_found),
          "| found if capital letters are ignored:", len(only_case), "| examples:", not_found[:5])
else:
    print("Skipped (RUN_LISTS = False)")


anime_id in the list but not in AnimeList: 0 ids, 0 rows | anime without any list row: 0 of 14,478

usernames in the list table: 283,045 | empty username cells: 0
distinct after lowercase: 283,045 | names that differ only by capital letters: 0
  examples: []
  with a space at the start or the end: 0
list usernames not found in UserList (exact text): 0 | found if capital letters are ignored: 0 | examples: []


In [33]:
# B5. my_watched_episodes against the episode count of the anime
if RUN_LISTS:
    print(f"rows whose anime has a known episode count (> 0): {ep_known:,} ({ep_known / rows:.1%})")
    print(f"my_watched_episodes above the episode count of the anime: {watched_over:,}")
    print(f"completed rows: {completed_rows:,} | with my_watched_episodes == 0: {completed_zero:,}"
          f" | below the episode count: {completed_less:,}")
else:
    print("Skipped (RUN_LISTS = False)")


rows whose anime has a known episode count (> 0): 79,375,560 (99.1%)
my_watched_episodes above the episode count of the anime: 207,305
completed rows: 50,445,526 | with my_watched_episodes == 0: 98,519 | below the episode count: 282,454


### Observations for part 7B

**How to read it.** The pass (B1) read all 80,076,112 rows in 27 chunks (about 4 minutes) and the next cells report what it counted.

**B2. `my_tags`: mostly empty, and the filled cells contain placeholders and spelling variants.**
- 4,760,289 rows have a tag (5.9%). 8,758 of them contain an HTML code.
- **53,395 cells are only a placeholder** (1.1% of the filled cells): `undefined` 33,194 (62% of them), `-` 6,194, `null` 1,957, `.` 1,423, `?` 1,422, `...` 1,162, `---` 1,002, `???` 694. `undefined` is even the most frequent value of the column. Why it appears we do not know.
- Spelling variants split the real tags: `Hentai` 31,295 against `hentai` 14,142, `Comedy` 13,664 against `comedy` 12,967. `anYme` (13,480) and `*` (9,237) are frequent and we do not know what they mean.
- **Limit:** a cell is counted as a whole. A cell such as `undefined, Hentai` is not counted, and the top list is approximate (a lower bound). To count single tags we must split each cell first (a step 4 decision).

**B3. Dates of the list table.**

| | `my_start_date` | `my_finish_date` |
|---|---|---|
| `0000-00-00` (no date) | 69,010,299 (86.2%) | 66,550,407 (83.1%) |
| Right format but not a real date, such as `2008-00-00` | 425,434 (0.5%) | 351,987 (0.4%) |
| Real dates | 10,640,379 (13.3%) | 13,173,718 (16.5%) |
| Earliest and latest real date | 1693-09-25 and 2115-04-22 | 1836-06-12 and 2101-09-26 |
| Before 1990 | 1,762 | 6,019 |
| After the crawl (2018-05-25) | 272 | 1,173 |
| Share of January 1 | 0.42% | 0.52% |

- **Most rows have no date at all.** Only 13% of the start dates and 16% of the finish dates are real, so any analysis of when users watched uses a small part of the table.
- **The five most frequent "not real" dates are all of the form `YYYY-00-00`**, a year with no month and no day (our reading: the user gave only the year). Impossible dates such as `2018-02-31` are mixed in this count; we did not separate them.
- **Absurd years exist:** a start date in 1693, a finish date in 1836, and dates up to 2115. They are typos. The 272 start dates and 1,173 finish dates after the crawl cannot be real viewing dates either (the crawl ended around 2018-05-25).
- **January 1 is not a default day here.** It is only 1.5 to 2 times its even share (0.27%), far from the 16 times seen in `birth_date`.
- **Some days are about 3 times too frequent** and they are not January 1: start dates on 5 to 8 April and 10 January (82,000 to 86,000 rows each, against about 29,000 if the days were spread evenly), finish dates on 27 to 30 March (100,000 to 114,000 each, against about 36,000) and 27 September. **Hypothesis, not tested:** anime seasons (new series start in early January and April, the winter season ends in late March), so these peaks would be real viewing habits and not placeholders.
- **The finish date is before the start date in 25,853 rows** (0.31% of the 8,264,175 rows that have two real dates).
- **`my_last_updated` is exactly 0 (1970-01-01) in 482,508 rows (0.6%)**, so it is a missing value. (No value lies between 1970 and November 2004, and the earliest is 0.) The latest value is 2018-05-22 08:01:08 and none is after the crawl, which agrees with the estimate of notebook 02.

**B4. `anime_id` and usernames of the list table (cleared).**
- Every `anime_id` of the list exists in `AnimeList` (0 missing), and every one of the 14,478 anime has at least one row.
- The list holds 283,045 usernames. None is empty, none has a space at the edge, none differs from another only by capital letters, and **all 283,045 are found exactly in `UserList`**. So the join on `username` is safe. (302,675 users minus 283,045 gives the 19,630 users without rows of notebook 03.)

**B5. Episodes watched against the episode count.**
- The episode count of the anime is known (above 0) for 79,375,560 rows (99.1%).
- **207,305 rows (0.26%) have more episodes watched than the anime has.**
- Of the 50,445,526 `completed` rows (63.0% of all rows), **98,519 have 0 episodes watched** (0.20%) and **282,454 have fewer episodes watched than the anime has** (0.56%).
- Possible reasons (hypotheses, not tested): the episode count of a long series changed after the user's entry, or the user did not update the counter. Either way `my_watched_episodes` is not exact for about 1% of the rows.

**What this changes.**
- Cleared (no action needed): numeric placeholders, usernames as a join key (both tables), orphan `anime_id`, duplicate anime, parse failures of the Python text columns, `score` range, `scored_by` against `members`.
- New problems for step 4: rows 23 to 30 of the summary below.
- **Still open:** what `undefined`, `anYme` and `*` mean in `my_tags`; whether the peaks on the 5th to 8th of April and 27th to 30th of March are seasons.


## Summary of step 3: the problems found, as input for step 4

Everything below was measured on the raw data. **Nothing was changed.** The last column is only a proposal, and the team decides in step 4.

| # | Problem | Size | Where | Proposal |
|---|---|---|---|---|
| 1 | Words that pandas hides as missing (`rating = None`, `location = N/A`, `null` username) | 544, 26 and 1 cells | Part 1 | Keep the safe load. Decide whether `None` and `N/A` count as missing |
| 2 | Placeholder words in the anime table (`source`, `broadcast`, `duration`, `aired_string`, `type`) | 4,210 / 2,241 / 405 / 223 / 30 | Part 2 | Turn into missing values, and report the counts |
| 3 | Anime columns that are mostly empty (`background`, `licensor`, `premiered`, `broadcast`) | 70% to 93% | Part 2 | Do not use as features |
| 4 | `score = 0` means no score | 354 anime (308 without raters, 46 unaired anime with few raters) | Parts 2 and 6 | Turn into missing values |
| 5 | Few raters | 4,789 anime (33.1%) have fewer than 100 | Part 2 | Choose a threshold and report the number removed |
| 6 | `rank` and `popularity` equal to 0, `episodes = 0` | 9, 9 and 504 anime | Parts 2 and 6 | Treat as missing values |
| 7 | `access_rank` is empty | 100% | Part 3 | Drop the column |
| 8 | Missing gender, birth date and location | 28.0% / 44.3% / 48.2% | Part 3 | Analyse the users who gave them, and say so |
| 9 | Impossible birth dates | 1,724 ages under 8 or over 80 (842 of the old ones were born in 1930, probably a default), 8 fake texts, years 1800 and 2137 | Parts 3 and 6 | Treat 1930 as missing, keep a plausible age range, report the number removed |
| 10 | `last_online` holds the year 1900 | 89,283 users (29.5%), all exactly in 1900 | Parts 3 and 6 | Do not use `last_online` as a date |
| 11 | `stats_mean_score = 0` means no score | 30,922 users | Part 3 | Turn into missing values |
| 12 | Impossible maxima in `user_days_spent_watching` and `stats_episodes` | 2,601,351 days and 243,794,900 episodes; 204 users above 100,000 episodes, 42 above 3,650 days | Parts 3 and 6 | Cap at the 99.9th percentile or remove |
| 13 | Declared counts differ from the list rows | 0.1% to 1.8% of the users per status; 6 declare 0 but have rows | Part 4 | Use the declared counts as a summary only |
| 14 | Users without rows | 19,630 users: 19,583 empty accounts, 47 with activity, 8 declaring anime | Part 5 | Drop the empty accounts from list analyses, report the 47 users with activity and the 8 declaring anime (some are in both groups) |
| 15 | Two usernames share each of 2 `user_id`s, with the same list stored twice | 1,059 duplicated list rows | Notebook 02, part 4 | Keep the name with the full profile |
| 16 | `duration` written in seconds | 313 anime (405 more are `Unknown`) | Part 6 | Read `sec` as minutes divided by 60; only the 405 `Unknown` are missing |
| 17 | Empty lists and empty dictionaries used as "no value" | `opening_theme` 9,784 (67.6%), `ending_theme` 8,807 (60.8%), `related` 4,515 (31.2%), `aired` 1,691 (11.7%) | Part 1a, 1b | Treat as missing (or as "none"). For the 1,691 empty `aired`, 1,468 have a year or a month in `aired_string` (part 1b): read the release year from `aired_string` |
| 18 | `broadcast` with a weekday and no time (`Sundays at Unknown`), and `Not scheduled once per week` | 78 and 65 anime | Part 1a | Treat the time as missing |
| 19 | Placeholders typed by users in `location` | 309 cells (0.2%), plus 78 inside longer texts | Part 1a | Build a list for `location`, and group the rest |
| 20 | `birth_date` with January 1 as a default day | 7,353 of 168,741 users = 4.36% (evenly spread: 0.27%, about 462 users); the share falls from 6.5% (born 1985) to 1.9% (born 2000) | Part 1b | Use the birth year only; do not use the month or the day |
| 21 | HTML codes such as `&#039;` and `&amp;` in text | 11 columns: `background` 316 (29.9%), `title_synonyms` 739 (8.3%), `title_english` 409 (7.1%), `title` 202, `studio` 130, `producer` 168, themes, `related`, `location` 40 | Part 1b | Decode every text column with `html.unescape` |
| 22 | Start dates stored with a filled-in year, month or day (the 2018 cluster on the 11th found in 1a) | Day 11 in general is not a default (457 of 12,787 start dates, mean 412 per day). The 2018 peaks on the 11th (`2018-05` 60, `2018-07` 27, `2018-10` 20) are dates that were filled in for a partial text (part 7, A5b); the total is 227 start dates and 146 end dates, part 7A, A5c. Days 21 (678) and 25 (808) stay open | Part 1b, 7A | Do not use the day or the year of a date whose `aired_string` is partial; read the release year from the text |
| 23 | `my_tags`: placeholders, spelling variants and HTML codes | 4,760,289 filled rows (5.9%); 53,395 placeholder cells (`undefined` 33,194); `Hentai` 31,295 against `hentai` 14,142; 8,758 with an HTML code | B2 | Treat the placeholders as empty, lowercase, decode, and split the cell into single tags before counting |
| 24 | `my_start_date` and `my_finish_date` equal to `0000-00-00` | 69,010,299 (86.2%) and 66,550,407 (83.1%) | B3 | Treat as missing; only about 13% to 16% of the rows have a real date |
| 25 | Dates with a year only (`YYYY-00-00`) or impossible years | 425,434 and 351,987 year-only; earliest year 1693 and 1836, latest 2115 and 2101; 272 and 1,173 after the crawl | B3 | Keep the year of the year-only dates; treat dates outside a plausible range as missing (decide the bounds) |
| 26 | Finish date before start date | 25,853 of 8,264,175 rows with two real dates (0.31%) | B3 | Treat both dates as unreliable in these rows |
| 27 | `my_last_updated` equal to 0 | 482,508 rows (0.6%) | B3 | Treat as missing |
| 28 | `location` spellings, spaces at the edge | 55,283 raw texts, 49,616 after normalising; 4,560 cells with an edge space | A2, A3 | Strip, lowercase and collapse spaces, then group places by rules |
| 29 | `aired` dates that were filled in for a partial `aired_string` (year 2018 for 82%, month May for every year-only text, days 9 to 12 for 96.5%), and the contradictions they create | 373 dates: 227 of the 12,787 start dates (1.8%) and 146 of the 12,287 end dates (1.2%). They explain all 16 "Finished Airing" with a start after the crawl, most of the 32 "end before start" (21 by eye) and 4 of the 6 "Currently Airing" with an end before the crawl | 7A (A5, A5b, A5c) | Flag these dates. Use a date only when the text is complete; for partial texts take the year from the text. Done for the 60 early-date anime of notebook 02: 54 of their dates are made up (notebook 02, part 4) |
| 30 | More episodes watched than the anime has; `completed` with few or no episodes | 207,305 rows; 98,519 completed with 0; 282,454 completed with fewer | B5 | Do not treat `my_watched_episodes` as exact; define a rule for these rows |
| 31 | Full dates in `aired` that are one year off around New Year (looks like the ISO week-numbering year) | 52 of 12,560 full starts (0.41%) and 19 of 6,467 full ends (0.29%); 29 to 31 December stored one year later, 1 to 2 January one year earlier | 7A (A5c) | Do not trust the year of a date between 29 December and 2 January; read it from the text |

**Cleared by part 7:** numeric placeholders, usernames (no capital-letter clash, no edge space, all list usernames found in `UserList`), orphan `anime_id`, duplicate anime, parse failures of the four Python text columns, `score` range, `scored_by` against `members`.

**What did not change.** The anime and user tables have no repeated key, the join on `username` has no orphan, and the declared counts agree with the list table for more than 98% of the users. The data is usable once the problems above are handled.
